# 부산 대중교통 노선 최적화 분석
**KAIA 2단계 - 버스 노선 효율성 분석 및 심야 노선 신설 제안**

---

## 분석 목차
- **Part 0.** 환경 설정 + 전체 데이터 로드
- **Part 1.** 하차 추정 알고리즘
- **Part 2.** 시나리오 1 - 버스 정류장 및 노선 변경
  - 2-1. 노선별 효율성 분석
  - 2-2. 비효율 노선 Top-K 선정
  - 2-3. 비효율 노선 변경 제안
- **Part 3.** 시나리오 2 - 심야 버스 노선 신설
  - 3-1. 심야 수요 분석
  - 3-2. 심야 서비스 공백 분석
  - 3-3. 심야 노선 제안
- **Part 4.** 종합 결과 저장

---
## Part 0. 환경 설정 + 전체 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString
from shapely.ops import unary_union, nearest_points
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import DBSCAN
from scipy.spatial import cKDTree
import networkx as nx
import os, zipfile, glob

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 기본 경로 (Windows PC)
BASE = r'C:\Users\HP\Desktop\작업 폴더\01.work\리버풀\데이터'
TARGET_CRS = 'EPSG:5179'
RESULT_DIR = f'{BASE}\\results_optimization'
os.makedirs(RESULT_DIR, exist_ok=True)

print('환경 설정 완료')
print(f'작업 경로: {BASE}')
print(f'결과 저장 경로: {RESULT_DIR}')

환경 설정 완료
작업 경로: C:\Users\HP\Desktop\작업 폴더\01.work\리버풀\데이터
결과 저장 경로: C:\Users\HP\Desktop\작업 폴더\01.work\리버풀\데이터\results_optimization


### 0.1 버스 노선 데이터

In [2]:
# 버스노선.csv 로드
bus_routes = pd.read_csv(f'{BASE}\\버스노선.csv', encoding='utf-8-sig')
bus_routes.columns = bus_routes.columns.str.strip().str.replace('\ufeff', '')

print(f'총 행수: {len(bus_routes):,}')
print(f'고유 노선 수: {bus_routes["노선번호"].nunique()}')
print(f'고유 정류소 수: {bus_routes["정류소ID"].nunique()}')
print(f'\n컬럼: {list(bus_routes.columns)}')
bus_routes.head()

총 행수: 24,576
고유 노선 수: 326
고유 정류소 수: 8510

컬럼: ['노선번호', '정류소순번', '정류소명', '시', '군.구', '동', 'GPS_X', 'GPX_Y', 'ARS ID', '정류소ID']


,노선번호,정류소순번,정류소명,시,군.구,동,GPS_X,GPX_Y,ARS ID,정류소ID
0,10,1,연제공용버스차고지,부산광역시,NaN,NaN,129.053194,35.185229,13179,50000001059
1,10,2,초읍고개,부산광역시,NaN,NaN,129.054089,35.185475,13183,50000002492
2,10,3,개인택시조합,부산광역시,연제구,거제2동,129.055711,35.186786,13132,21130521003
3,10,4,부산의료원 정문,부산광역시,NaN,NaN,129.058971,35.187956,13186,50000003510
4,10,5,부산의료원,부산광역시,연제구,거제2동,129.060579,35.188257,13123,21130521007


In [3]:
# 노선 유형 분류
def classify_route(name):
    name = str(name)
    if '심야' in name:
        return '심야버스'
    elif '아침' in name or '출' in name:
        return '출퇴근버스'
    elif name.startswith(('1001','1002','1003','1004')):
        return '급행버스'
    else:
        try:
            int(name)
            return '시내버스'
        except:
            return '마을버스'

route_list = bus_routes.groupby('노선번호').agg(
    정류소수=('정류소순번', 'max'),
    시작정류소=('정류소명', 'first'),
    종료정류소=('정류소명', 'last')
).reset_index()

route_list['노선유형'] = route_list['노선번호'].apply(classify_route)

print('=== 노선 유형별 현황 ===')
print(route_list['노선유형'].value_counts())
print(f'\n노선당 평균 정류소 수: {route_list["정류소수"].mean():.1f}')

=== 노선 유형별 현황 ===
노선유형
마을버스     178
시내버스     125
심야버스      15
급행버스       4
출퇴근버스      4
Name: count, dtype: int64

노선당 평균 정류소 수: 75.4


### 0.2 버스 노선 GeoDataFrame 생성

In [4]:
# 버스노선 정류소를 GeoDataFrame으로 변환
bus_routes_gdf = gpd.GeoDataFrame(
    bus_routes,
    geometry=gpd.points_from_xy(bus_routes['GPS_X'], bus_routes['GPX_Y']),
    crs='EPSG:4326'
).to_crs(TARGET_CRS)

print(f'버스 노선 GeoDataFrame: {len(bus_routes_gdf):,}건, CRS: {bus_routes_gdf.crs}')

# 노선별 LineString 생성 (정류소 순서대로 연결)
route_lines = {}
for route_id, group in bus_routes_gdf.sort_values(['노선번호','정류소순번']).groupby('노선번호'):
    coords = list(group.geometry.apply(lambda g: (g.x, g.y)))
    if len(coords) >= 2:
        route_lines[route_id] = LineString(coords)

route_lines_gdf = gpd.GeoDataFrame(
    {'노선번호': list(route_lines.keys())},
    geometry=list(route_lines.values()),
    crs=TARGET_CRS
)
print(f'노선 LineString 수: {len(route_lines_gdf)}')

버스 노선 GeoDataFrame: 24,576건, CRS: EPSG:5179
노선 LineString 수: 326


### 0.3 공간 데이터 로드

In [5]:
# 버스 정류소 SHP
bus_stops_shp = gpd.read_file(f'{BASE}\\부산광역시_버스 정류소 정보(SHP)_20250121\\tl_bus_station_info.shp')
if bus_stops_shp.crs.to_epsg() != 5179:
    bus_stops_shp = bus_stops_shp.to_crs(TARGET_CRS)
print(f'버스 정류소 SHP: {len(bus_stops_shp):,}건, CRS: {bus_stops_shp.crs}')

# 50m 격자
grid_50m = gpd.read_file(f'{BASE}\\50m_grid\\busan_50cell_v21.shp')
if grid_50m.crs.to_epsg() != 5179:
    grid_50m = grid_50m.to_crs(TARGET_CRS)
print(f'50m 격자: {len(grid_50m):,}건, CRS: {grid_50m.crs}')

# 집계구 경계
census = gpd.read_file(f'{BASE}\\집계구경계\\집계구경계.shp')
if census.crs.to_epsg() != 5179:
    census = census.to_crs(TARGET_CRS)
print(f'집계구: {len(census):,}건, CRS: {census.crs}')

버스 정류소 SHP: 8,522건, CRS: EPSG:5179
50m 격자: 310,241건, CRS: EPSG:5179
집계구: 6,802건, CRS: EPSG:5179


### 0.4 차량 네트워크 (노드-링크) 로드

In [6]:
# ZIP 파일 해제 (필요시)
network_zip = f'{BASE}\\차량네트워크\\car_walk_node_202507.zip'
network_dir = f'{BASE}\\차량네트워크\\car_walk_node_202507'

if not os.path.exists(network_dir):
    with zipfile.ZipFile(network_zip, 'r') as z:
        z.extractall(f'{BASE}\\차량네트워크')
    print('ZIP 해제 완료')

# car_link.shp 로드
car_link_path = glob.glob(f'{network_dir}\\**\\car_link_202507.shp', recursive=True)
if not car_link_path:
    car_link_path = glob.glob(f'{BASE}\\차량네트워크\\**\\car_link_202507.shp', recursive=True)
car_link = gpd.read_file(car_link_path[0])
print(f'car_link: {len(car_link):,}건, CRS: {car_link.crs}')
print(f'  컬럼: {list(car_link.columns[:15])}')

# car_nod.shp 로드
car_nod_path = glob.glob(f'{network_dir}\\**\\car_node_202507.shp', recursive=True)
if not car_nod_path:
    car_nod_path = glob.glob(f'{BASE}\\차량네트워크\\**\\car_node_202507.shp', recursive=True)
car_node = gpd.read_file(car_nod_path[0])
print(f'\ncar_node: {len(car_node):,}건, CRS: {car_node.crs}')

# walk_link.shp 로드
walk_link_path = glob.glob(f'{network_dir}\\**\\walk_link_202507.shp', recursive=True)
if not walk_link_path:
    walk_link_path = glob.glob(f'{BASE}\\차량네트워크\\**\\walk_link_202507.shp', recursive=True)
walk_link = gpd.read_file(walk_link_path[0])
print(f'\nwalk_link: {len(walk_link):,}건, CRS: {walk_link.crs}')

# CRS 통일
for gdf_name in ['car_link', 'car_node', 'walk_link']:
    gdf = eval(gdf_name)
    if gdf.crs and gdf.crs.to_epsg() != 5179:
        exec(f'{gdf_name} = {gdf_name}.to_crs(TARGET_CRS)')
        print(f'{gdf_name} CRS 변환 완료')

print(f'\ncar_link 주요 컬럼 샘플:')
print(car_link[['LINK_ID', 'ST_ND_ID', 'ED_ND_ID', 'LENGTH']].head())


car_link: 108,983건, CRS: EPSG:5179
  컬럼: ['LINK_ID', 'TLINKIDP1', 'TLINKIDN1', 'ST_ND_ID', 'ED_ND_ID', 'LENGTH', 'ST_DIR', 'ED_DIR', 'ROAD_CATE', 'ROADLEVEL', 'ROADSTATE', 'LINK_CATE', 'LINK_FACIL', 'ROAD_NO', 'ONEWAY']

car_node: 88,332건, CRS: EPSG:5179

walk_link: 185,143건, CRS: EPSG:5179

car_link 주요 컬럼 샘플:
  LINK_ID ST_ND_ID ED_ND_ID LENGTH
0       1        1        2   1489
1       1        1       20    299
2     100       23      122    140
3    1000      738      739    128
4    1001      355      738     68


### 0.5 도로 네트워크 그래프 구축

In [7]:
# NetworkX 그래프 구축 (차량 네트워크)
print('도로 네트워크 그래프 구축 중...')
G_car = nx.DiGraph()

# 노드 추가 (좌표 정보 포함)
for _, row in car_node.iterrows():
    node_id = str(row['NODE_ID'])
    G_car.add_node(node_id, x=row.geometry.x, y=row.geometry.y)

# 링크 추가 (가중치 = 실제 길이)
for _, row in car_link.iterrows():
    st = str(row['ST_ND_ID'])
    ed = str(row['ED_ND_ID'])
    length = float(row['LENGTH']) if pd.notna(row['LENGTH']) else row.geometry.length
    link_id = str(row['LINK_ID'])
    G_car.add_edge(st, ed, weight=length, link_id=link_id)

    # 양방향 추가 (blanest 컬럼 확인 - 일방통행 여부)
    if 'blanest' in car_link.columns:
        if str(row.get('blanest', '')) != '1':  # 일방통행이 아닌 경우 역방향도 추가
            G_car.add_edge(ed, st, weight=length, link_id=link_id)
    else:
        G_car.add_edge(ed, st, weight=length, link_id=link_id)

print(f'그래프 구축 완료: 노드 {G_car.number_of_nodes():,}개, 엣지 {G_car.number_of_edges():,}개')

# 노드 좌표 KDTree 생성 (최근접 노드 탐색용)
node_ids = list(G_car.nodes())
node_coords = np.array([(G_car.nodes[n].get('x', 0), G_car.nodes[n].get('y', 0)) for n in node_ids])
node_tree = cKDTree(node_coords)


def find_nearest_node(x, y, tree=node_tree, ids=node_ids):
    """좌표 (x, y)에서 가장 가까운 도로 네트워크 노드 ID 반환"""
    _, idx = tree.query([x, y])
    return ids[idx]


def shortest_path_length(src_x, src_y, dst_x, dst_y, graph=G_car):
    """두 좌표 사이 도로 네트워크 기반 최단 경로 길이(m) 반환"""
    src_node = find_nearest_node(src_x, src_y)
    dst_node = find_nearest_node(dst_x, dst_y)
    try:
        return nx.shortest_path_length(graph, src_node, dst_node, weight='weight')
    except nx.NetworkXNoPath:
        return np.nan


def shortest_path_nodes(src_x, src_y, dst_x, dst_y, graph=G_car):
    """두 좌표 사이 도로 네트워크 기반 최단 경로 노드 리스트 반환"""
    src_node = find_nearest_node(src_x, src_y)
    dst_node = find_nearest_node(dst_x, dst_y)
    try:
        return nx.shortest_path(graph, src_node, dst_node, weight='weight')
    except nx.NetworkXNoPath:
        return []


print('최단경로 함수 준비 완료')


도로 네트워크 그래프 구축 중...
그래프 구축 완료: 노드 23,716개, 엣지 186,410개
최단경로 함수 준비 완료


### 0.6 노선별 실제 경로 길이 산출 (노드-링크 기반)

In [ ]:
# 노선별 실제 경로 길이 산출 (연속 정류소 간 도로 네트워크 최단경로 합산)
print('노선별 실제 경로 길이 산출 중... (노드-링크 기반)')

route_lengths = {}
route_link_sets = {}  # 노선별 사용 링크 ID 집합 (중복도 계산용)

for route_id, group in bus_routes_gdf.sort_values(['노선번호','정류소순번']).groupby('노선번호'):
    stops = group.reset_index(drop=True)
    total_len = 0.0
    link_set = set()
    
    for i in range(len(stops) - 1):
        sx, sy = stops.iloc[i].geometry.x, stops.iloc[i].geometry.y
        ex, ey = stops.iloc[i+1].geometry.x, stops.iloc[i+1].geometry.y
        
        # 최단 경로 길이
        seg_len = shortest_path_length(sx, sy, ex, ey)
        if pd.isna(seg_len):
            # 경로 없으면 직선 거리 대체
            seg_len = np.sqrt((ex - sx)**2 + (ey - sy)**2)
        total_len += seg_len
        
        # 사용 링크 수집
        path = shortest_path_nodes(sx, sy, ex, ey)
        for j in range(len(path) - 1):
            edge_data = G_car.get_edge_data(path[j], path[j+1])
            if edge_data and 'link_id' in edge_data:
                link_set.add(edge_data['link_id'])
    
    route_lengths[route_id] = total_len / 1000.0  # m -> km
    route_link_sets[route_id] = link_set

route_list['노선길이_km'] = route_list['노선번호'].map(route_lengths)
print(f'노선 길이 산출 완료')
print(f'평균 노선 길이: {route_list["노선길이_km"].mean():.2f} km')
print(f'최소: {route_list["노선길이_km"].min():.2f} km, 최대: {route_list["노선길이_km"].max():.2f} km')

노선별 실제 경로 길이 산출 중... (노드-링크 기반)


### 0.7 POI 데이터

In [ ]:
# POI 데이터 로드
poi = pd.read_csv(f'{BASE}\\BUSAN_POI\\부산광역시.csv', encoding='utf-8')
poi.columns = poi.columns.str.strip().str.replace('\ufeff', '')

poi_valid = poi.dropna(subset=['x(EPSG:5179)', 'y(EPSG:5179)'])
poi_gdf = gpd.GeoDataFrame(
    poi_valid,
    geometry=gpd.points_from_xy(
        poi_valid['x(EPSG:5179)'].astype(float),
        poi_valid['y(EPSG:5179)'].astype(float)
    ),
    crs=TARGET_CRS
)
print(f'POI GeoDataFrame: {len(poi_gdf):,}건')

### 0.8 생활인구 및 카드소비 데이터

In [ ]:
# 생활인구 - 성연령별 (메모리 효율적으로 처리)
print('생활인구 데이터 로드 및 집계 중...')

# 청크 단위로 읽으면서 바로 집계
chunk_size = 500000
grid_pop_list = []

for chunk in pd.read_csv(f'{BASE}\\생활인구&소비\\kt_d_living_emd_p_202507.csv', sep='|', chunksize=chunk_size):
    # 성연령 컬럼만 선택해서 합산
    pop_cols = [c for c in chunk.columns if c.startswith(('h_m_', 'h_w_', 'w_m_', 'w_w_', 'v_m_', 'v_w_'))]
    chunk['총생활인구'] = chunk[pop_cols].sum(axis=1)

    # 격자별 집계
    chunk_agg = chunk.groupby('id').agg(총인구합=('총생활인구', 'sum'), 데이터수=('총생활인구', 'count')).reset_index()
    grid_pop_list.append(chunk_agg)

# 청크별 집계 결과를 다시 합산
grid_pop_combined = pd.concat(grid_pop_list, ignore_index=True)
grid_pop = grid_pop_combined.groupby('id').agg(
    총인구합=('총인구합', 'sum'),
    데이터수=('데이터수', 'sum')
).reset_index()
grid_pop['평균생활인구'] = grid_pop['총인구합'] / grid_pop['데이터수']
grid_pop = grid_pop[['id', '평균생활인구']]

print(f'생활인구(성연령별) 집계 완료: {len(grid_pop):,}개 격자')

# 생활인구 - 시간대별 (심야 0~5시)
print('심야 생활인구 집계 중...')
grid_night_pop_list = []

for chunk in pd.read_csv(f'{BASE}\\생활인구&소비\\kt_d_living_tme_p_202507.csv', sep='|', chunksize=chunk_size):
    # 심야(0~5시) 컬럼만 선택
    night_cols = [c for c in chunk.columns if any(c.endswith(f'_{h:02d}') for h in range(0, 6))]
    if night_cols:
        chunk['심야생활인구'] = chunk[night_cols].sum(axis=1)
        chunk_agg = chunk.groupby('id').agg(총심야인구합=('심야생활인구', 'sum'), 데이터수=('심야생활인구', 'count')).reset_index()
        grid_night_pop_list.append(chunk_agg)

if grid_night_pop_list:
    grid_night_pop_combined = pd.concat(grid_night_pop_list, ignore_index=True)
    grid_night_pop = grid_night_pop_combined.groupby('id').agg(
        총심야인구합=('총심야인구합', 'sum'),
        데이터수=('데이터수', 'sum')
    ).reset_index()
    grid_night_pop['평균심야인구'] = grid_night_pop['총심야인구합'] / grid_night_pop['데이터수']
    grid_night_pop = grid_night_pop[['id', '평균심야인구']]
    print(f'심야 생활인구 집계 완료: {len(grid_night_pop):,}개 격자')

# 카드소비 데이터 (더 작으니 일반 로드)
card_consume_emd = pd.read_csv(f'{BASE}\\생활인구&소비\\bccd_d_emd_p_202507.csv', sep='|')
card_consume_tme = pd.read_csv(f'{BASE}\\생활인구&소비\\bccd_d_tme_p_202507.csv', sep='|')
print(f'카드소비(성연령별): {len(card_consume_emd):,}건')
print(f'카드소비(시간대별): {len(card_consume_tme):,}건')

print(f'\n격자별 데이터 집계 완료')


### 0.9 사회형평 데이터

In [ ]:
# 다문화지원센터
multicultural = pd.read_csv(f'{BASE}\\사회형평\\부산광역시_다문화지원센터 현황.csv', encoding='utf-8-sig')
print(f'다문화지원센터: {len(multicultural):,}건')
print(f'  컬럼: {list(multicultural.columns)}')

# 장애인 등록 현황
disabled = pd.read_csv(f'{BASE}\\사회형평\\시군구별_장애정도별_성별_등록장애인수_20260330100131.csv', encoding='utf-8-sig')
print(f'\n등록장애인수: {len(disabled):,}건')
print(f'  컬럼: {list(disabled.columns)[:10]}')
disabled.head()

### 0.10 교통카드 거래내역 로드 + 파생변수 생성

In [ ]:
# 교통카드 거래내역 (parquet)
card_columns = [
    'date', 'card_type', 'card_number', 'seq', 'in_out',
    'transport_id', 'transport_name', 'station_id', 'station_name',
    'device_id', 'transaction_id', 'payment_type', 'time',
    'passenger_type', 'passenger_detail', 'passenger_count', 'station_seq'
]

card_df = pd.read_parquet(f'{BASE}\\A001_202507\\A001_202507_processed.parquet')

# 컬럼명 보정
if card_df.columns.tolist()[0] == 0 or 'Unnamed' in str(card_df.columns[0]):
    if len(card_df.columns) == 18:
        card_df = card_df.iloc[:, 1:]
    card_df.columns = card_columns

print(f'=== 교통카드 거래내역 ===')
print(f'행수: {len(card_df):,}')
print(f'기간: {card_df["date"].min()} ~ {card_df["date"].max()}')

# 파생변수 생성
# 1) 환승횟수 및 승하차 구분
card_df['환승횟수'] = card_df['in_out'].astype(str).str.split('-').str[0].astype(int)
card_df['승하차'] = card_df['in_out'].astype(str).str.split('-').str[1].map({'0':'승차','1':'하차'})

# 2) 시간 파싱
card_df['time_str'] = card_df['time'].astype(str).str.zfill(4)
card_df['시간대'] = card_df['time_str'].str[:2].astype(int)

# 3) 수단 구분
card_df['수단'] = card_df['transaction_id'].astype(str).str.strip().apply(
    lambda x: '지하철' if x == '00000000' or x == '0' else '버스'
)

# 4) 탑승순서
card_df['탑승순서'] = card_df['환승횟수'] + 1

# 5) 심야 여부 (0~5시)
card_df['심야여부'] = card_df['시간대'].isin([0, 1, 2, 3, 4, 5])

print(f'\n수단별 건수:')
print(card_df['수단'].value_counts())
print(f'\n승하차별 건수:')
print(card_df['승하차'].value_counts())

# 버스만 필터
bus_card = card_df[card_df['수단'] == '버스'].copy()
print(f'\n버스 거래내역: {len(bus_card):,}건')
print(f'  승차: {(bus_card["승하차"]=="승차").sum():,}건')
print(f'  하차: {(bus_card["승하차"]=="하차").sum():,}건')
print(f'  하차 결측률: {(1 - (bus_card["승하차"]=="하차").sum() / (bus_card["승하차"]=="승차").sum()) * 100:.1f}%')

### 0.11 데이터 ID 체계 진단 및 매핑 테이블 구축
교통카드 `station_id`와 버스노선 `정류소ID`/`ARS ID`의 형식 차이를 확인하고 매핑합니다.

In [ ]:
# ===========================================================
# 데이터 ID 체계 진단 + station_id 매핑 테이블 구축
# ===========================================================
print('=== 데이터 ID 체계 진단 ===')

# 1) 카드데이터 station_id 샘플
card_sids = bus_card['station_id'].dropna().astype(str).unique()
print(f'카드데이터 station_id 고유값: {len(card_sids)}개')
print(f'  샘플: {card_sids[:10]}')
print(f'  길이 분포: {pd.Series([len(s) for s in card_sids[:10000]]).value_counts().sort_index().to_dict()}')

# 2) 버스노선 CSV 정류소ID / ARS ID 샘플
route_sids = bus_routes['정류소ID'].dropna().astype(str).unique()
route_ars = bus_routes['ARS ID'].dropna().astype(str).unique()
print(f'\n버스노선 정류소ID 고유값: {len(route_sids)}개, 샘플: {route_sids[:5]}')
print(f'버스노선 ARS ID 고유값: {len(route_ars)}개, 샘플: {route_ars[:5]}')

# 3) SHP bstopid 샘플
shp_sids = bus_stops_shp['bstopid'].dropna().astype(str).unique()
print(f'\nSHP bstopid 고유값: {len(shp_sids)}개, 샘플: {shp_sids[:5]}')

# 4) 교집합 확인 -> 어떤 ID 체계끼리 매칭되는지
card_set = set(card_sids)
route_id_set = set(route_sids)
route_ars_set = set(route_ars)
shp_set = set(shp_sids)

print(f'\n=== 매칭 교집합 ===')
print(f'card ∩ 정류소ID: {len(card_set & route_id_set)}')
print(f'card ∩ ARS ID:   {len(card_set & route_ars_set)}')
print(f'card ∩ SHP bstopid: {len(card_set & shp_set)}')
print(f'정류소ID ∩ bstopid: {len(route_id_set & shp_set)}')
print(f'ARS ID ∩ bstopid: {len(route_ars_set & shp_set)}')

# 5) 최적 매핑 결정
matches = {
    'card↔정류소ID': len(card_set & route_id_set),
    'card↔ARS_ID': len(card_set & route_ars_set),
    'card↔bstopid': len(card_set & shp_set),
}
best_match = max(matches, key=matches.get)
print(f'\n최적 매핑: {best_match} ({matches[best_match]}개 매칭)')

# 6) 매핑 테이블 구축
# 버스노선 CSV에서 (정류소ID, ARS ID, 정류소명) 매핑
stop_mapping = bus_routes[['정류소ID', 'ARS ID', '정류소명']].drop_duplicates()
stop_mapping['정류소ID'] = stop_mapping['정류소ID'].astype(str)
stop_mapping['ARS ID'] = stop_mapping['ARS ID'].astype(str)

# card station_id -> 정류소ID 변환 딕셔너리 구축
if best_match == 'card↔ARS_ID':
    # ARS ID 기반 매핑
    ars_to_stopid = stop_mapping.groupby('ARS ID')['정류소ID'].first().to_dict()
    bus_card['정류소ID_mapped'] = bus_card['station_id'].astype(str).map(ars_to_stopid)
    STATION_KEY = '정류소ID_mapped'
    print(f'ARS ID 기반 매핑 적용. 매핑률: {bus_card["정류소ID_mapped"].notna().mean()*100:.1f}%')
elif best_match == 'card↔정류소ID':
    bus_card['정류소ID_mapped'] = bus_card['station_id'].astype(str)
    STATION_KEY = '정류소ID_mapped'
    print('정류소ID 직접 매칭')
elif best_match == 'card↔bstopid':
    # bstopid -> 정류소ID 매핑 (SHP와 노선CSV 연결 필요)
    # SHP의 bstopid가 노선CSV의 어느 것과 매칭되는지 확인
    if len(shp_set & route_id_set) > len(shp_set & route_ars_set):
        bus_card['정류소ID_mapped'] = bus_card['station_id'].astype(str)
    else:
        bstop_to_stopid = dict(zip(
            bus_stops_shp['bstopid'].astype(str),
            bus_stops_shp['bstopid'].astype(str)  # SHP ID 그대로
        ))
        bus_card['정류소ID_mapped'] = bus_card['station_id'].astype(str).map(bstop_to_stopid)
    STATION_KEY = '정류소ID_mapped'
    print(f'bstopid 기반 매핑 적용')
else:
    # 매칭 안 되면 station_name 기반 폴백
    name_to_stopid = stop_mapping.groupby('정류소명')['정류소ID'].first().to_dict()
    bus_card['정류소ID_mapped'] = bus_card['station_name'].astype(str).map(name_to_stopid)
    STATION_KEY = '정류소ID_mapped'
    print(f'정류소명 기반 폴백 매핑. 매핑률: {bus_card["정류소ID_mapped"].notna().mean()*100:.1f}%')

# transport_name ↔ 노선번호 매칭 확인
card_routes = set(bus_card['transport_name'].dropna().astype(str).unique())
csv_routes = set(bus_routes['노선번호'].dropna().astype(str).unique())
print(f'\ntransport_name↔노선번호 매칭: {len(card_routes & csv_routes)}/{len(csv_routes)}')
if len(card_routes & csv_routes) < len(csv_routes) * 0.8:
    # 매칭률이 80% 미만이면 진단
    print('  미매칭 노선번호 샘플:', list(csv_routes - card_routes)[:10])
    print('  미매칭 transport_name 샘플:', list(card_routes - csv_routes)[:10])

# bus_boarding_only 재정의 (매핑된 정류소ID 포함)
bus_boarding_only = bus_card[bus_card['승하차'] == '승차'].copy()
print(f'\nbus_boarding_only 재구축: {len(bus_boarding_only):,}건')
print(f'  정류소ID 매핑률: {bus_boarding_only[STATION_KEY].notna().mean()*100:.1f}%')


---
## Part 1. 하차 추정 알고리즘

버스 교통카드 데이터에서 하차 정보가 누락된 거래를 추정합니다.

**추정 로직:**
1. 동일 카드, 동일 날짜의 거래를 시간순 정렬
2. 승차 후 하차 기록이 없으면: 다음 승차 정류소 = 이전 하차 추정
3. 마지막 탑승의 하차 정류소: 첫 승차 정류소로 추정 (귀가 가정)

**검증:** 하차 기록이 있는 데이터를 train/test로 분리하여 성능 확인

In [ ]:
# ================================================================
# Part 1-A. 통행사슬 구축 + 노선-정류소 매핑
# ================================================================
from scipy.spatial import cKDTree as cKDTree2

print('=== 하차 추정 데이터 구축 ===')

# 1) 노선별 정류소 순서/좌표 매핑
route_stop_info = {}      # {노선: {정류소ID: {순번, x, y}}}
route_sorted_seqs = {}    # {노선: [(순번, 정류소ID), ...]}
route_terminus_map = {}   # {노선: 종점정류소ID}
route_stop_seq_map = {}   # {(노선, 정류소ID): 순번}

for route_id, group in bus_routes_gdf.sort_values(['노선번호','정류소순번']).groupby('노선번호'):
    rid = str(route_id)
    stops = group.reset_index(drop=True)
    info = {}
    seq_list = []
    for _, row in stops.iterrows():
        sid = str(row['정류소ID'])
        seq_num = int(row['정류소순번'])
        info[sid] = {'순번': seq_num, 'x': row.geometry.x, 'y': row.geometry.y}
        seq_list.append((seq_num, sid))
        route_stop_seq_map[(rid, sid)] = seq_num
    route_stop_info[rid] = info
    seq_list.sort()
    route_sorted_seqs[rid] = seq_list
    route_terminus_map[rid] = seq_list[-1][1] if seq_list else None

# 정류소 좌표
stop_coords = {}
for _, row in bus_routes_gdf.iterrows():
    sid = str(row['정류소ID'])
    if sid not in stop_coords:
        stop_coords[sid] = (row.geometry.x, row.geometry.y)
for _, row in bus_stops_shp.iterrows():
    sid = str(row['bstopid'])
    if sid not in stop_coords:
        stop_coords[sid] = (row.geometry.x, row.geometry.y)

# 노선별 정류소 KDTree
route_stop_trees = {}
route_stop_ids_arr = {}
for rid, info in route_stop_info.items():
    sids = list(info.keys())
    coords_arr = np.array([(info[s]['x'], info[s]['y']) for s in sids])
    if len(coords_arr) >= 1:
        route_stop_trees[rid] = cKDTree2(coords_arr)
        route_stop_ids_arr[rid] = sids

seq_map_flat = {f"{k[0]}|{k[1]}": v for k, v in route_stop_seq_map.items()}

# ★ FIX: 매핑된 정류소ID로도 좌표 접근 가능하도록
# (stop_coords의 key는 정류소ID 형식이므로, 카드데이터 station_id → 정류소ID 변환 후 사용)

print(f'노선별 정류소 매핑: {len(route_stop_info)}개 노선')
print(f'정류소 좌표 매핑: {len(stop_coords)}개 정류소')

# 2) 통행사슬 구축 (merge 기반)
print('\n통행사슬 구축 중...')
bus_sorted = bus_card.sort_values(['card_number', 'date', 'time']).reset_index(drop=True)

# ★ FIX: seq가 아닌 환승횟수로 승차/하차 매칭
boarding = bus_sorted[bus_sorted['승하차'] == '승차'][
    ['card_number','date','환승횟수','station_id','station_name','transport_name',
     'time','시간대','passenger_type', STATION_KEY]
].drop_duplicates(subset=['card_number','date','환승횟수'], keep='first').copy()
boarding.columns = [
    'card_number','date','환승횟수','승차정류소_raw','승차정류소명','노선명',
    '승차시간','시간대','승객유형','승차정류소'
]

alighting = bus_sorted[bus_sorted['승하차'] == '하차'][
    ['card_number','date','환승횟수','station_id','station_name', STATION_KEY]
].drop_duplicates(subset=['card_number','date','환승횟수'], keep='first').copy()
alighting.columns = ['card_number','date','환승횟수','하차정류소_raw','하차정류소명','하차정류소']

trip_pairs = boarding.merge(alighting, on=['card_number','date','환승횟수'], how='left')
trip_pairs['하차있음'] = trip_pairs['하차정류소'].notna()

trip_pairs['요일'] = pd.to_datetime(trip_pairs['date'].astype(str)).dt.dayofweek

def time_bin(h):
    if h < 6: return '심야'
    elif h < 10: return '출근'
    elif h < 14: return '오전'
    elif h < 18: return '오후'
    elif h < 22: return '저녁'
    else: return '야간'

trip_pairs['시간대빈'] = trip_pairs['시간대'].apply(time_bin)

# 노선 내 승/하차 순번
# ★ FIX: 노선명(transport_name)으로 매핑 (seq_map_flat의 key도 동일 체계)
trip_pairs['_승키'] = trip_pairs['노선명'].astype(str) + '|' + trip_pairs['승차정류소'].astype(str)
trip_pairs['_하키'] = trip_pairs['노선명'].astype(str) + '|' + trip_pairs['하차정류소'].astype(str)
trip_pairs['승차노선순번'] = trip_pairs['_승키'].map(seq_map_flat)
trip_pairs['하차노선순번'] = trip_pairs['_하키'].map(seq_map_flat)
trip_pairs.drop(columns=['_승키','_하키'], inplace=True)

# 다음 승차 정류소 / 첫 승차 정류소
trip_pairs = trip_pairs.sort_values(['card_number','date','승차시간']).reset_index(drop=True)
trip_pairs['다음승차정류소'] = trip_pairs.groupby(['card_number','date'])['승차정류소'].shift(-1)
trip_pairs['첫승차정류소'] = trip_pairs.groupby(['card_number','date'])['승차정류소'].transform('first')

# 이동정류소수 (하차있는 건)
trip_pairs['이동정류소수'] = np.where(
    trip_pairs['하차있음'] & trip_pairs['승차노선순번'].notna() & trip_pairs['하차노선순번'].notna(),
    trip_pairs['하차노선순번'] - trip_pairs['승차노선순번'],
    np.nan
)

print(f'통행 쌍: {len(trip_pairs):,}건')
print(f'  하차 있음: {trip_pairs["하차있음"].sum():,}건 ({trip_pairs["하차있음"].mean()*100:.1f}%)')
print(f'  승차노선순번 매핑률: {trip_pairs["승차노선순번"].notna().mean()*100:.1f}%')

In [ ]:
# ================================================================
# Part 1-B. 다단계 추정 모델 구축 함수
# ================================================================

def build_estimation_models(train_trips):
    '''학습 데이터로 추정 통계 + ML 모델 구축'''
    has = train_trips[train_trips['하차있음']].copy()
    has['card_number'] = has['card_number'].astype(str)
    has['노선명'] = has['노선명'].astype(str)
    has['승차정류소'] = has['승차정류소'].astype(str)
    has['하차정류소'] = has['하차정류소'].astype(str)

    # --- 1. 개인 이력: (카드, 노선, 승차정류소) → 최빈 하차정류소 ---
    personal_counts = has.groupby(['card_number','노선명','승차정류소','하차정류소']).size().reset_index(name='cnt')
    personal_mode = personal_counts.sort_values('cnt', ascending=False).drop_duplicates(
        subset=['card_number','노선명','승차정류소'], keep='first'
    )[['card_number','노선명','승차정류소','하차정류소']].copy()
    personal_mode.columns = ['card_number','노선명','승차정류소','개인최빈하차']
    print(f'  개인 이력 매핑: {len(personal_mode):,}건')

    # --- 2. (노선, 승차정류소, 시간대빈) → 최빈 하차정류소 ---
    rst_counts = has.groupby(['노선명','승차정류소','시간대빈','하차정류소']).size().reset_index(name='cnt')
    rst_mode = rst_counts.sort_values('cnt', ascending=False).drop_duplicates(
        subset=['노선명','승차정류소','시간대빈'], keep='first'
    )[['노선명','승차정류소','시간대빈','하차정류소']].copy()
    rst_mode.columns = ['노선명','승차정류소','시간대빈','통계하차_시간']
    print(f'  (노선,정류소,시간대) 매핑: {len(rst_mode):,}건')

    # --- 3. (노선, 승차정류소) → 최빈 하차정류소 ---
    rs_counts = has.groupby(['노선명','승차정류소','하차정류소']).size().reset_index(name='cnt')
    rs_mode = rs_counts.sort_values('cnt', ascending=False).drop_duplicates(
        subset=['노선명','승차정류소'], keep='first'
    )[['노선명','승차정류소','하차정류소']].copy()
    rs_mode.columns = ['노선명','승차정류소','통계하차']
    print(f'  (노선,정류소) 매핑: {len(rs_mode):,}건')

    # --- 4. ML: 이동 정류소 수 예측 ---
    ml_model = None
    ml_features = ['시간대','요일','환승횟수','승차노선순번','노선정류소수','승객유형_enc']
    ml_data = has.dropna(subset=['이동정류소수','승차노선순번']).copy()
    ml_data = ml_data[ml_data['이동정류소수'] > 0]  # 음수/0 제거

    if len(ml_data) > 1000:
        # 노선 정류소수
        route_n_stops = {}
        for rid, seqs in route_sorted_seqs.items():
            route_n_stops[rid] = len(seqs)
        ml_data['노선정류소수'] = ml_data['노선명'].map(route_n_stops).fillna(30)

        # 승객유형 인코딩
        pt_map = {str(v): i for i, v in enumerate(ml_data['승객유형'].unique())}
        ml_data['승객유형_enc'] = ml_data['승객유형'].astype(str).map(pt_map).fillna(0).astype(int)

        try:
            from lightgbm import LGBMRegressor
            model_cls = LGBMRegressor(n_estimators=300, max_depth=8, learning_rate=0.05,
                                      num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                                      min_child_samples=50, verbose=-1, n_jobs=-1)
            print('  ML: LightGBM 사용')
        except ImportError:
            from sklearn.ensemble import GradientBoostingRegressor
            model_cls = GradientBoostingRegressor(n_estimators=200, max_depth=6,
                                                  learning_rate=0.05, subsample=0.8)
            print('  ML: GradientBoosting (fallback) 사용')

        X_ml = ml_data[ml_features].values
        y_ml = ml_data['이동정류소수'].values

        model_cls.fit(X_ml, y_ml)
        ml_model = {'model': model_cls, 'features': ml_features,
                    'pt_map': pt_map, 'route_n_stops': route_n_stops}
        print(f'  ML 학습 완료: {len(ml_data):,}건')
    else:
        print('  ML 학습 데이터 부족, skip')

    return {
        'personal': personal_mode,
        'route_stop_time': rst_mode,
        'route_stop': rs_mode,
        'ml': ml_model
    }


def find_nearest_route_stop(route_id, target_x, target_y):
    '''노선 내에서 (target_x, target_y)에 가장 가까운 정류소 ID 반환'''
    rid = str(route_id)
    if rid not in route_stop_trees:
        return None
    tree = route_stop_trees[rid]
    sids = route_stop_ids_arr[rid]
    _, idx = tree.query([target_x, target_y])
    return sids[idx]


def find_route_stop_after(route_id, boarding_seq, target_x, target_y):
    '''노선 내에서 boarding_seq 이후 정류소 중 target에 가장 가까운 것'''
    rid = str(route_id)
    if rid not in route_stop_info:
        return None
    info = route_stop_info[rid]
    candidates = [(sid, d['x'], d['y']) for sid, d in info.items() if d['순번'] > boarding_seq]
    if not candidates:
        return route_terminus_map.get(rid)
    dists = [np.sqrt((c[1]-target_x)**2 + (c[2]-target_y)**2) for c in candidates]
    return candidates[np.argmin(dists)][0]


def estimate_alighting_multi(trips_df, models):
    '''다단계 하차 추정'''
    result = trips_df.copy()
    result['추정하차정류소'] = result['하차정류소'].copy()
    result['추정하차정류소'] = result['추정하차정류소'].astype(object)
    result['추정방법'] = np.where(result['하차있음'], '실제하차', None)
    need_est = result['추정방법'].isna()

    # ---- 1단계: 개인 이력 ----
    merged = result[need_est].merge(
        models['personal'],
        on=['card_number','노선명','승차정류소'], how='left'
    )
    filled = merged['개인최빈하차'].notna()
    if filled.any():
        idx = result.index[need_est][filled.values]
        result.loc[idx, '추정하차정류소'] = merged.loc[filled, '개인최빈하차'].values
        result.loc[idx, '추정방법'] = '1_개인이력'
    need_est = result['추정방법'].isna()
    print(f'  1단계(개인이력): {(result["추정방법"]=="1_개인이력").sum():,}건')

    # ---- 2단계: (노선, 승차정류소, 시간대) 통계 ----
    merged = result[need_est].merge(
        models['route_stop_time'],
        on=['노선명','승차정류소','시간대빈'], how='left'
    )
    filled = merged['통계하차_시간'].notna()
    if filled.any():
        idx = result.index[need_est][filled.values]
        result.loc[idx, '추정하차정류소'] = merged.loc[filled, '통계하차_시간'].values
        result.loc[idx, '추정방법'] = '2_통계_시간대'
    need_est = result['추정방법'].isna()
    print(f'  2단계(통계_시간대): {(result["추정방법"]=="2_통계_시간대").sum():,}건')

    # ---- 3단계: (노선, 승차정류소) 통계 ----
    merged = result[need_est].merge(
        models['route_stop'],
        on=['노선명','승차정류소'], how='left'
    )
    filled = merged['통계하차'].notna()
    if filled.any():
        idx = result.index[need_est][filled.values]
        result.loc[idx, '추정하차정류소'] = merged.loc[filled, '통계하차'].values
        result.loc[idx, '추정방법'] = '3_통계_노선정류소'
    need_est = result['추정방법'].isna()
    print(f'  3단계(통계_노선정류소): {(result["추정방법"]=="3_통계_노선정류소").sum():,}건')

    # ---- 4단계: ML 모델 ----
    ml = models.get('ml')
    if ml is not None and need_est.any():
        ml_subset = result[need_est].copy()
        ml_subset['노선정류소수'] = ml_subset['노선명'].astype(str).map(ml['route_n_stops']).fillna(30)
        ml_subset['승객유형_enc'] = ml_subset['승객유형'].astype(str).map(ml['pt_map']).fillna(0).astype(int)

        has_seq = ml_subset['승차노선순번'].notna()
        if has_seq.any():
            X_pred = ml_subset.loc[has_seq, ml['features']].fillna(0).values
            pred_delta = ml['model'].predict(X_pred)
            pred_delta = np.round(pred_delta).astype(int).clip(1, None)

            est_stops = []
            for i, (_, row) in enumerate(ml_subset[has_seq].iterrows()):
                rid = str(row['노선명'])
                b_seq = int(row['승차노선순번'])
                target_seq = b_seq + pred_delta[i]
                # 노선 내에서 target_seq에 가장 가까운 정류소
                seqs = route_sorted_seqs.get(rid, [])
                if seqs:
                    best = min(seqs, key=lambda s: abs(s[0] - target_seq))
                    est_stops.append(best[1])
                else:
                    est_stops.append(None)

            idx = result.index[need_est][has_seq.values]
            vals = pd.Series(est_stops, index=idx)
            filled_ml = vals.notna()
            if filled_ml.any():
                result.loc[idx[filled_ml], '추정하차정류소'] = vals[filled_ml].values
                result.loc[idx[filled_ml], '추정방법'] = '4_ML모델'
        need_est = result['추정방법'].isna()
        print(f'  4단계(ML): {(result["추정방법"]=="4_ML모델").sum():,}건')

    # ---- 5단계: 다음 승차 + 노선 제약 ----
    if need_est.any():
        subset = result[need_est].copy()
        has_next = subset['다음승차정류소'].notna() & subset['승차노선순번'].notna()
        est_5 = []
        for _, row in subset[has_next].iterrows():
            next_sid = str(row['다음승차정류소'])
            if next_sid in stop_coords:
                tx, ty = stop_coords[next_sid]
                est = find_route_stop_after(str(row['노선명']), int(row['승차노선순번']), tx, ty)
                est_5.append(est)
            else:
                est_5.append(None)
        idx = result.index[need_est][has_next.values]
        vals = pd.Series(est_5, index=idx)
        filled_5 = vals.notna()
        if filled_5.any():
            result.loc[idx[filled_5], '추정하차정류소'] = vals[filled_5].values
            result.loc[idx[filled_5], '추정방법'] = '5_다음승차_노선제약'
        need_est = result['추정방법'].isna()
        print(f'  5단계(다음승차_노선제약): {(result["추정방법"]=="5_다음승차_노선제약").sum():,}건')

    # ---- 6단계: 노선 종점 폴백 ----
    if need_est.any():
        subset = result[need_est]
        terminus = subset['노선명'].astype(str).map(route_terminus_map)
        filled_6 = terminus.notna()
        if filled_6.any():
            idx = subset.index[filled_6.values]
            result.loc[idx, '추정하차정류소'] = terminus[filled_6].values
            result.loc[idx, '추정방법'] = '6_종점폴백'
        need_est = result['추정방법'].isna()
        print(f'  6단계(종점폴백): {(result["추정방법"]=="6_종점폴백").sum():,}건')

    # 남은 미추정
    print(f'  미추정: {need_est.sum():,}건')
    return result


print('추정 함수 정의 완료')

In [ ]:
# ================================================================
# Part 1-C. Train/Test 분리 및 성능 검증
# ================================================================
print('=== 하차 추정 알고리즘 검증 ===')

# 시간 기반 분리: 마지막 날을 테스트로
dates_sorted = sorted(trip_pairs['date'].unique())
if len(dates_sorted) >= 7:
    train_dates = dates_sorted[:-2]
    test_dates = dates_sorted[-2:]
elif len(dates_sorted) >= 2:
    train_dates = dates_sorted[:-1]
    test_dates = dates_sorted[-1:]
else:
    # fallback: 랜덤 80/20
    train_dates = dates_sorted
    test_dates = dates_sorted

print(f'학습 기간: {min(train_dates)} ~ {max(train_dates)} ({len(train_dates)}일)')
print(f'테스트 기간: {min(test_dates)} ~ {max(test_dates)} ({len(test_dates)}일)')

train_trips = trip_pairs[trip_pairs['date'].isin(train_dates)].copy()
test_trips = trip_pairs[trip_pairs['date'].isin(test_dates)].copy()

# 테스트: 하차 있는 건만 평가 대상
test_has_alight = test_trips[test_trips['하차있음']].copy()
test_eval = test_has_alight.copy()
test_eval['실제하차정류소'] = test_eval['하차정류소'].copy()

# 하차 마스킹 (2건 이상 그룹에서 마지막 제외)
group_counts = test_eval.groupby(['card_number','date']).cumcount(ascending=False)
# 모든 하차 기록을 마스킹해서 추정 테스트 (마지막 건 포함)
test_eval['하차정류소'] = None
test_eval['하차있음'] = False

print(f'테스트 샘플: {len(test_eval):,}건')

# 모델 구축 (train만)
print('\n모델 구축 중...')
models = build_estimation_models(train_trips)

# 추정 실행
print('\n추정 실행 중...')
estimated = estimate_alighting_multi(test_eval, models)

# ---- 성능 평가 ----
eval_data = estimated[estimated['추정방법'] != '실제하차'].copy()
eval_data = eval_data[eval_data['실제하차정류소'].notna()].copy()

# Exact match
eval_data['정답여부'] = eval_data['추정하차정류소'].astype(str) == eval_data['실제하차정류소'].astype(str)

# 거리 기반 평가
def calc_distance(row):
    est = str(row['추정하차정류소'])
    real = str(row['실제하차정류소'])
    if est in stop_coords and real in stop_coords:
        ex, ey = stop_coords[est]
        rx, ry = stop_coords[real]
        return np.sqrt((ex-rx)**2 + (ey-ry)**2)
    return np.nan

eval_data['거리오차_m'] = eval_data.apply(calc_distance, axis=1)

# 결과 출력
accuracy_exact = eval_data['정답여부'].mean() * 100
has_dist = eval_data['거리오차_m'].notna()
accuracy_300m = (eval_data.loc[has_dist, '거리오차_m'] <= 300).mean() * 100 if has_dist.any() else 0
accuracy_500m = (eval_data.loc[has_dist, '거리오차_m'] <= 500).mean() * 100 if has_dist.any() else 0
accuracy_1km = (eval_data.loc[has_dist, '거리오차_m'] <= 1000).mean() * 100 if has_dist.any() else 0

print(f'\n{"="*50}')
print(f'=== 하차 추정 성능 (최종) ===')
print(f'{"="*50}')
print(f'추정 대상: {len(eval_data):,}건')
print(f'Exact Match 정확도: {accuracy_exact:.1f}%')
print(f'300m 이내 정확도:   {accuracy_300m:.1f}%')
print(f'500m 이내 정확도:   {accuracy_500m:.1f}%')
print(f'1km 이내 정확도:    {accuracy_1km:.1f}%')
print(f'평균 거리오차:       {eval_data["거리오차_m"].mean():.0f}m')
print(f'중앙값 거리오차:     {eval_data["거리오차_m"].median():.0f}m')

# 추정방법별 성능
print(f'\n추정방법별 성능:')
method_perf = eval_data.groupby('추정방법').agg(
    건수=('정답여부', 'count'),
    Exact정확도=('정답여부', 'mean'),
    평균거리오차=('거리오차_m', 'mean'),
    중앙값거리오차=('거리오차_m', 'median')
).round(3)
method_perf['Exact정확도'] = (method_perf['Exact정확도'] * 100).round(1)
method_perf['평균거리오차'] = method_perf['평균거리오차'].round(0)
method_perf['중앙값거리오차'] = method_perf['중앙값거리오차'].round(0)
print(method_perf.to_string())

# 추정 커버리지
coverage = eval_data['추정하차정류소'].notna().mean() * 100
print(f'\n추정 커버리지: {coverage:.1f}%')

In [ ]:
# ================================================================
# Part 1-D. 전체 데이터에 하차 추정 적용
# ================================================================
print('전체 통행 데이터에 하차 추정 적용 중...')
print('전체 데이터로 모델 재구축...')

# 전체 데이터(하차 있는 건)로 모델 재구축
models_full = build_estimation_models(trip_pairs)

# 추정 적용
print('\n추정 적용 중...')
trip_pairs_estimated = estimate_alighting_multi(trip_pairs, models_full)

print(f'\n추정 결과:')
print(trip_pairs_estimated['추정방법'].value_counts())
print(f'\n추정 커버리지: {trip_pairs_estimated["추정하차정류소"].notna().mean()*100:.1f}%')

In [ ]:
# 하차 추정 알고리즘 구현 (개선 버전 v2 - 노선 기반 강화)

def estimate_alighting_improved(trips_df, bus_routes_gdf):
    """
    개선된 하차 추정 알고리즘 v2
    
    규칙 우선순위:
    1. 다음 승차가 같은 노선 = 하차 후 재탑승 → 다음 승차 정류소
    2. 마지막 통행 + 첫 승차 정류소 자주 이용 = 귀가 → 첫 승차 정류소
    3. 통계 기반 (같은 노선, 같은 승차 정류소에서 가장 많이 하차한 곳)
    4. 노선 종점 (운행 시간 고려)
    """
    print('하차 추정 시작...')
    result = trips_df.copy()
    
    # 추정 관련 컬럼 초기화
    result['추정하차정류소'] = result['하차정류소']
    result['추정방법'] = '실제하차'
    result.loc[~result['하차있음'], '추정방법'] = None
    
    # 1. 노선별 정류소 맵 구축 (정류소 검증용)
    # 주의: trip_pairs의 '노선명'과 bus_routes_gdf의 '노선번호'를 매칭
    route_stops = {}
    for route, group in bus_routes_gdf.groupby('노선번호'):
        route_stops[str(route)] = set(group['정류소ID'].astype(str))
    
    # 2. 통계 구축: (노선, 승차정류소) -> 하차정류소 빈도
    stats_df = result[result['하차있음']].copy()
    route_station_stats = stats_df.groupby(['노선명', '승차정류소', '하차정류소']).size().reset_index(name='빈도')
    route_station_stats = route_station_stats.sort_values(['노선명', '승차정류소', '빈도'], ascending=[True, True, False])
    
    # 각 (노선, 승차정류소) 조합의 top-1 하차 정류소 추출
    top_alighting = route_station_stats.groupby(['노선명', '승차정류소']).first().reset_index()
    top_alighting_dict = {}
    for _, row in top_alighting.iterrows():
        key = (row['노선명'], row['승차정류소'])
        top_alighting_dict[key] = row['하차정류소']
    
    print(f'노선별 통계 구축 완료: {len(route_station_stats["노선명"].unique())}개 노선')
    
    # 3. 카드별 첫 승차 정류소 통계 (귀가지 추정용)
    first_boarding_stats = result.groupby('card_number').agg(
        첫승차정류소_최빈값=('승차정류소', lambda x: x.mode()[0] if len(x.mode()) > 0 else None)
    ).to_dict()['첫승차정류소_최빈값']
    
    # 4. 노선별 종점 정보
    route_terminals = {}
    for route, group in bus_routes_gdf.groupby('노선번호'):
        sorted_stops = group.sort_values('정류소순번')
        route_terminals[str(route)] = {
            'first': sorted_stops.iloc[0]['정류소ID'],
            'last': sorted_stops.iloc[-1]['정류소ID']
        }
    
    # 5. 추정 로직 적용
    estimated_count = 0
    method_counts = {}
    
    # 카드-날짜별 그룹 처리
    for (card, date), group in result.groupby(['card_number', 'date']):
        indices = group.sort_values('승차시간').index.tolist()
        
        for i, idx in enumerate(indices):
            # 이미 하차 정보가 있으면 스킵
            if result.loc[idx, '하차있음']:
                continue
            
            estimated = False
            boarding_station = result.loc[idx, '승차정류소']
            boarding_route = result.loc[idx, '노선명']
            boarding_time = result.loc[idx, '승차시간']
            
            # 규칙 1: 다음 승차가 같은 노선 → 다음 승차 정류소
            if i + 1 < len(indices):
                next_idx = indices[i + 1]
                next_route = result.loc[next_idx, '노선명']
                next_station = result.loc[next_idx, '승차정류소']
                next_time = result.loc[next_idx, '승차시간']
                
                # 같은 노선이고 시간차가 합리적이면 (5분~120분)
                if next_route == boarding_route and pd.notna(next_time) and pd.notna(boarding_time):
                    time_diff = next_time - boarding_time
                    # 자정 넘어가는 경우 처리
                    if time_diff < 0:
                        time_diff += 2400
                    
                    if 5 <= time_diff <= 120:
                        # 노선에 속한 정류소인지 검증
                        if boarding_route in route_stops and str(next_station) in route_stops[boarding_route]:
                            result.loc[idx, '추정하차정류소'] = str(next_station)
                            result.loc[idx, '추정방법'] = '같은노선재탑승'
                            estimated = True
                            method_counts['같은노선재탑승'] = method_counts.get('같은노선재탑승', 0) + 1
            
            # 규칙 2: 마지막 통행 → 첫 승차 정류소 (귀가 추정)
            if not estimated and i == len(indices) - 1:
                home_station = first_boarding_stats.get(card)
                if pd.notna(home_station) and home_station != boarding_station:
                    # 노선에 속한 정류소인지 검증
                    if boarding_route in route_stops and str(home_station) in route_stops[boarding_route]:
                        result.loc[idx, '추정하차정류소'] = str(home_station)
                        result.loc[idx, '추정방법'] = '귀가추정'
                        estimated = True
                        method_counts['귀가추정'] = method_counts.get('귀가추정', 0) + 1
            
            # 규칙 3: 통계 기반 (같은 노선, 같은 승차 정류소)
            if not estimated:
                key = (boarding_route, boarding_station)
                if key in top_alighting_dict:
                    result.loc[idx, '추정하차정류소'] = str(top_alighting_dict[key])
                    result.loc[idx, '추정방법'] = '통계기반추정(같은노선)'
                    estimated = True
                    method_counts['통계기반추정(같은노선)'] = method_counts.get('통계기반추정(같은노선)', 0) + 1
            
            # 규칙 4: 노선 종점 (시간대 고려)
            if not estimated and boarding_route in route_terminals:
                terminals = route_terminals[boarding_route]
                # 심야시간(22시~5시)이면 종점 가능성 높음
                hour = boarding_time // 100
                if hour >= 22 or hour < 5:
                    result.loc[idx, '추정하차정류소'] = str(terminals['last'])
                    result.loc[idx, '추정방법'] = '노선종점추정'
                    estimated = True
                    method_counts['노선종점추정'] = method_counts.get('노선종점추정', 0) + 1
            
            if estimated:
                estimated_count += 1
    
    print(f'\n추정 완료: {estimated_count:,}건 추가 추정')
    if method_counts:
        print('추정 방법별 건수:')
        for method, count in sorted(method_counts.items(), key=lambda x: -x[1]):
            print(f'  {method}: {count:,}건')
    
    return result

print('개선된 하차 추정 함수 v2 정의 완료 (같은 노선 재탑승 + 귀가 추정)')


---
## Part 2. 시나리오 1 - 버스 정류장 및 노선 변경

### 2-1. 노선별 효율성 분석

In [ ]:
# ===========================================================
# 지표 1: 수요효율 = 노선별 승차량 / 노선길이(km)
# ===========================================================
print('=== 지표 1: 수요효율 ===')

# 노선별 일평균 승차량 (버스만)
# bus_boarding_only는 Cell 0.11에서 이미 정의됨 (정류소ID_mapped 포함)
# transport_name으로 노선 매칭
route_demand = bus_boarding_only.groupby('transport_name').agg(
    총승차량=('card_number', 'count'),
    운행일수=('date', 'nunique')
).reset_index()
route_demand['일평균승차량'] = route_demand['총승차량'] / route_demand['운행일수']

# 노선번호와 transport_name 매칭
route_demand.rename(columns={'transport_name': '노선번호'}, inplace=True)

# route_list와 병합
efficiency = route_list.merge(route_demand[['노선번호','일평균승차량','총승차량']], on='노선번호', how='left')
efficiency['일평균승차량'] = efficiency['일평균승차량'].fillna(0)

# 수요효율 = 일평균 승차량 / 노선길이(km)
efficiency['수요효율'] = efficiency['일평균승차량'] / efficiency['노선길이_km'].replace(0, np.nan)

print(f'수요효율 통계:')
print(efficiency['수요효율'].describe().round(2))

In [ ]:
# ===========================================================
# 지표 2: 커버리지 효율 = 서비스 영역(400m 버퍼) 내 생활인구 / 정류소 수
# ===========================================================
print('=== 지표 2: 커버리지 효율 ===')

# 격자 데이터에 생활인구 결합 (타입 맞춤)
grid_pop['id'] = grid_pop['id'].astype(str)  # int64 -> str로 변환
grid_with_pop = grid_50m.merge(grid_pop, on='id', how='left')
grid_with_pop['평균생활인구'] = grid_with_pop['평균생활인구'].fillna(0)

# 노선별 커버리지 산출
coverage_results = []

for route_id, group in bus_routes_gdf.groupby('노선번호'):
    # 정류소 400m 버퍼
    stops_buffer = group.geometry.buffer(400)
    route_buffer = unary_union(stops_buffer)

    # 버퍼 내 격자의 생활인구 합산
    covered_grids = grid_with_pop[grid_with_pop.geometry.intersects(route_buffer)]
    pop_covered = covered_grids['평균생활인구'].sum()

    n_stops = len(group)

    coverage_results.append({
        '노선번호': route_id,
        '커버인구': pop_covered,
        '정류소수_실제': n_stops,
        '커버리지효율': pop_covered / n_stops if n_stops > 0 else 0
    })

coverage_df = pd.DataFrame(coverage_results)
efficiency = efficiency.merge(coverage_df[['노선번호', '커버리지효율', '커버인구']], on='노선번호', how='left')

print(f'커버리지 효율 통계:')
print(efficiency['커버리지효율'].describe().round(2))


In [ ]:
# ===========================================================
# 지표 3: 중복도 = 다른 노선과 겹치는 링크 비율
# ===========================================================
print('=== 지표 3: 노선 중복도 ===')

# 전체 링크 사용 현황 (어떤 노선이 어떤 링크를 사용하는지)
all_links = set()
link_usage_count = {}  # link_id -> 사용 노선 수

for route_id, links in route_link_sets.items():
    for link_id in links:
        if link_id not in link_usage_count:
            link_usage_count[link_id] = 0
        link_usage_count[link_id] += 1
        all_links.add(link_id)

# 노선별 중복도 = (2개 이상 노선이 사용하는 링크 수) / (해당 노선의 전체 링크 수)
overlap_results = []
for route_id, links in route_link_sets.items():
    if len(links) == 0:
        overlap_ratio = 0
    else:
        shared = sum(1 for l in links if link_usage_count.get(l, 0) >= 2)
        overlap_ratio = shared / len(links)
    overlap_results.append({'노선번호': route_id, '중복도': overlap_ratio})

overlap_df = pd.DataFrame(overlap_results)
efficiency = efficiency.merge(overlap_df, on='노선번호', how='left')

print(f'중복도 통계:')
print(efficiency['중복도'].describe().round(3))

In [ ]:
# ===========================================================
# 지표 4: 수요-공급 균형 = 정류소별 승차량 변동계수 (CV)
# ===========================================================
print('=== 지표 4: 수요-공급 균형 (CV) ===')

# 노선별 정류소별 승차량
stop_demand = bus_boarding_only.groupby(['transport_name', 'station_id']).size().reset_index(name='승차량')

# 노선별 CV 계산
cv_results = []
for route_id, group in stop_demand.groupby('transport_name'):
    mean_val = group['승차량'].mean()
    std_val = group['승차량'].std()
    cv = std_val / mean_val if mean_val > 0 else 0
    cv_results.append({'노선번호': route_id, 'CV': cv})

cv_df = pd.DataFrame(cv_results)
efficiency = efficiency.merge(cv_df, on='노선번호', how='left')

print(f'CV 통계:')
print(efficiency['CV'].describe().round(3))

In [ ]:
# ===========================================================
# 지표 5: 시간대 효율 = 첨두 vs 비첨두 이용률 비율
# ===========================================================
print('=== 지표 5: 시간대 효율 ===')

# 첨두시간: 7-9시, 17-19시
bus_boarding_only['첨두'] = bus_boarding_only['시간대'].isin([7, 8, 9, 17, 18, 19])

time_eff_results = []
for route_id, group in bus_boarding_only.groupby('transport_name'):
    peak = group[group['첨두']].shape[0]
    offpeak = group[~group['첨두']].shape[0]
    # 비첨두 대비 첨두 비율 (높을수록 첨두 집중 = 비효율적 가능성)
    ratio = peak / offpeak if offpeak > 0 else 0
    time_eff_results.append({'노선번호': route_id, '첨두비율': ratio})

time_eff_df = pd.DataFrame(time_eff_results)
efficiency = efficiency.merge(time_eff_df, on='노선번호', how='left')

print(f'첨두비율 통계:')
print(efficiency['첨두비율'].describe().round(3))

In [ ]:
# ===========================================================
# 지표 6: 사회형평 점수 = 노선 커버 영역 내 취약계층 비율
# ===========================================================
print('=== 지표 6: 사회형평 점수 ===')

# 교통카드에서 노인 비율 (passenger_type 기반)
# passenger_type 확인
print('승객유형 분포:')
print(bus_boarding_only['passenger_type'].value_counts().head(10))

# 노선별 노인 비율 (passenger_type에서 노인 관련 유형)
# 일반적으로: 1=일반, 2=어린이, 3=청소년, 4=노인 등
equity_results = []
for route_id, group in bus_boarding_only.groupby('transport_name'):
    total = len(group)
    # passenger_detail에서 노인 관련 키워드 또는 passenger_type으로 판별
    # ★ FIX: passenger_type이 문자열('어른,', '청소년,' 등)이므로 문자열 기반 판별
    elderly_keywords_pt = ['노인', '경로', '65세', '어르신']
    elderly_keywords_detail = ['노인', '경로', '65세', '어르신', '장애']
    
    # passenger_type에서 노인 키워드 검색
    elderly_by_type = group[group['passenger_type'].astype(str).str.contains(
        '|'.join(elderly_keywords_pt), na=False, case=False
    )].shape[0]
    
    # passenger_detail에서도 검색
    elderly_by_detail = group[group['passenger_detail'].astype(str).str.contains(
        '|'.join(elderly_keywords_detail), na=False, case=False
    )].shape[0]
    
    elderly_count = max(elderly_by_type, elderly_by_detail)
    
    # 취약계층 확장: 어린이도 포함
    child_count = group[group['passenger_type'].astype(str).str.contains(
        '어린이', na=False
    )].shape[0]
    elderly_count = elderly_count + child_count  # 취약계층 = 노인 + 어린이
    
    elderly_ratio = elderly_count / total if total > 0 else 0
    equity_results.append({'노선번호': route_id, '사회형평점수': elderly_ratio})

equity_df = pd.DataFrame(equity_results)
efficiency = efficiency.merge(equity_df, on='노선번호', how='left')

print(f'\n사회형평 점수 통계:')
print(efficiency['사회형평점수'].describe().round(4))

In [ ]:
# ===========================================================
# 지표 7: 카드소비 연계 = 노선 주변 경제활동 밀도
# ===========================================================
print('=== 지표 7: 카드소비 연계 ===')

# 격자별 카드소비 합산
consume_by_grid = card_consume_emd.groupby('id').agg(
    총소비금액=('amt', 'sum'),
    총소비건수=('cnt', 'sum')
).reset_index()

consume_by_grid['id'] = consume_by_grid['id'].astype(str)  # int64 -> str로 변환
grid_with_consume = grid_50m.merge(consume_by_grid, on='id', how='left')
grid_with_consume['총소비금액'] = grid_with_consume['총소비금액'].fillna(0)

# 노선별 주변 경제활동 밀도
econ_results = []
for route_id, group in bus_routes_gdf.groupby('노선번호'):
    stops_buffer = unary_union(group.geometry.buffer(400))
    covered = grid_with_consume[grid_with_consume.geometry.intersects(stops_buffer)]
    total_consume = covered['총소비금액'].sum()
    n_stops = len(group)
    econ_density = total_consume / n_stops if n_stops > 0 else 0
    econ_results.append({'노선번호': route_id, '경제활동밀도': econ_density})

econ_df = pd.DataFrame(econ_results)
efficiency = efficiency.merge(econ_df, on='노선번호', how='left')

print(f'경제활동밀도 통계:')
print(efficiency['경제활동밀도'].describe().round(0))


In [ ]:
# ===========================================================
# 종합 효율 점수 산출 (가중합)
# ===========================================================
print('=== 종합 효율 점수 ===')

# 정규화 함수 (Min-Max)
def normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# 정규화 (방향 주의: 높을수록 좋은 것 vs 나쁜 것)
efficiency['수요효율_norm'] = normalize(efficiency['수요효율'].fillna(0))  # 높을수록 좋음
efficiency['커버리지효율_norm'] = normalize(efficiency['커버리지효율'].fillna(0))  # 높을수록 좋음
efficiency['중복도_norm'] = 1 - normalize(efficiency['중복도'].fillna(0))  # 낮을수록 좋음 -> 반전
efficiency['CV_norm'] = 1 - normalize(efficiency['CV'].fillna(0))  # 낮을수록 좋음 -> 반전
efficiency['첨두비율_norm'] = normalize(efficiency['첨두비율'].fillna(0))  # 높은 첨두비율 = 수요 집중 (중립적)
efficiency['사회형평_norm'] = normalize(efficiency['사회형평점수'].fillna(0))  # 높을수록 좋음
efficiency['경제활동_norm'] = normalize(efficiency['경제활동밀도'].fillna(0))  # 높을수록 좋음

# 가중치 설정 (총합 = 1.0)
weights = {
    '수요효율_norm': 0.25,       # 수요 대비 공급
    '커버리지효율_norm': 0.20,    # 인구 커버리지
    '중복도_norm': 0.15,         # 중복 최소화
    'CV_norm': 0.10,            # 수요 균형
    '첨두비율_norm': 0.05,       # 시간대 효율
    '사회형평_norm': 0.15,       # 사회형평
    '경제활동_norm': 0.10        # 경제활동
}

# 종합 점수 산출
efficiency['종합효율점수'] = sum(
    efficiency[col] * w for col, w in weights.items()
)

# 랭킹
efficiency['효율순위'] = efficiency['종합효율점수'].rank(ascending=True, method='min').astype(int)

print(f'종합 효율 점수 통계:')
print(efficiency['종합효율점수'].describe().round(4))

print(f'\n상위 10개 효율 노선:')
print(efficiency.nlargest(10, '종합효율점수')[['노선번호','노선유형','일평균승차량','노선길이_km','종합효율점수','효율순위']].to_string(index=False))

print(f'\n하위 10개 비효율 노선:')
print(efficiency.nsmallest(10, '종합효율점수')[['노선번호','노선유형','일평균승차량','노선길이_km','종합효율점수','효율순위']].to_string(index=False))

### 2-2. 비효율 노선 Top-K 선정

In [ ]:
# ===========================================================
# 비효율 노선 Top-K 선정
# ===========================================================
K = 10  # 조절 가능한 파라미터

# 심야버스 제외 (심야는 Part 3에서 별도 분석)
non_night = efficiency[efficiency['노선유형'] != '심야버스'].copy()
inefficient_topk = non_night.nsmallest(K, '종합효율점수').copy()

print(f'=== 비효율 노선 Top-{K} (심야 제외) ===')

# 비효율 사유 분석
norm_cols = ['수요효율_norm', '커버리지효율_norm', '중복도_norm', 'CV_norm', '사회형평_norm', '경제활동_norm']
col_names = ['수요효율', '커버리지효율', '중복도', '수요균형(CV)', '사회형평', '경제활동']

for _, row in inefficient_topk.iterrows():
    print(f'\n--- {row["노선번호"]} ({row["노선유형"]}) ---')
    print(f'  종합 점수: {row["종합효율점수"]:.4f} (순위: {row["효율순위"]})')
    print(f'  일평균 승차량: {row["일평균승차량"]:.0f}명, 노선길이: {row["노선길이_km"]:.1f}km')
    
    # 가장 낮은 지표 3개 찾기
    scores = {name: row[col] for col, name in zip(norm_cols, col_names)}
    worst = sorted(scores.items(), key=lambda x: x[1])[:3]
    print(f'  주요 비효율 사유:')
    for name, score in worst:
        print(f'    - {name}: {score:.3f}')

# 결과 테이블
display_cols = ['노선번호', '노선유형', '일평균승차량', '노선길이_km', '수요효율', '커버리지효율',
                '중복도', 'CV', '사회형평점수', '경제활동밀도', '종합효율점수', '효율순위']
print(f'\n=== 비효율 Top-{K} 요약 테이블 ===')
print(inefficient_topk[display_cols].to_string(index=False))

### 2-3. 비효율 노선 변경 제안

In [ ]:
# ===========================================================
# 비효율 노선별 정류장 제거/이동 제안 (v3 - 통계 기준 + 노드링크 기반)
# ===========================================================
print('=== 비효율 노선 정류장 제거/이동 제안 ===')

total_days = bus_card['date'].nunique()

# ----- 생활인구 격자 KDTree (이동 후보 위치 탐색용) -----
# grid_pop: 격자별 생활인구 (Cell 19에서 집계됨)
# grid_50m: 50m 격자 GDF
grid_merged = grid_50m.copy()
grid_merged['cx'] = grid_merged.geometry.centroid.x
grid_merged['cy'] = grid_merged.geometry.centroid.y

# 생활인구 데이터가 있으면 병합
if 'grid_pop' in dir() and grid_pop is not None:
    # grid_pop의 키 컬럼 확인 후 병합
    pop_col = [c for c in grid_pop.columns if '인구' in c or 'pop' in c.lower()]
    grid_key = grid_merged.columns[0]  # 첫 컬럼이 보통 격자 ID
    
    if hasattr(grid_pop, 'columns'):
        grid_pop_key = grid_pop.columns[0]
        grid_merged = grid_merged.merge(
            grid_pop[[grid_pop_key] + pop_col[:1]], 
            left_on=grid_key, right_on=grid_pop_key, how='left'
        )
        POP_COL = pop_col[0] if pop_col else None
    else:
        POP_COL = None
else:
    POP_COL = None

# 격자별 심야 생활인구도 있으면 병합
if 'grid_night_pop' in dir() and grid_night_pop is not None:
    pass  # 이미 grid_merged에 포함될 수 있음

# ----- 사회형평 점수 격자화 -----
# 장애인, 다문화 등 취약시설 300m 내 격자에 가산점
equity_points = []
if 'multicultural' in dir() and multicultural is not None:
    try:
        mc_gdf = gpd.GeoDataFrame(
            multicultural,
            geometry=gpd.points_from_xy(
                multicultural.iloc[:, -2] if multicultural.select_dtypes(include=[np.number]).shape[1] >= 2 else [0]*len(multicultural),
                multicultural.iloc[:, -1] if multicultural.select_dtypes(include=[np.number]).shape[1] >= 2 else [0]*len(multicultural)
            ), crs='EPSG:4326'
        ).to_crs(TARGET_CRS)
        equity_points.append(mc_gdf)
    except:
        pass

print(f'운행일수: {total_days}일')
print(f'사회형평 시설 포인트: {sum(len(ep) for ep in equity_points)}개')

# ----- 도로 네트워크 노드 KDTree (이동 좌표 탐색용) -----
car_node_coords = np.array([(G_car.nodes[n].get('x',0), G_car.nodes[n].get('y',0)) for n in G_car.nodes()])
car_node_ids = list(G_car.nodes())
car_node_tree = cKDTree(car_node_coords)

# ----- 정류소 KDTree -----
stop_shp_coords = np.column_stack([bus_stops_shp.geometry.x, bus_stops_shp.geometry.y])
stop_shp_tree = cKDTree(stop_shp_coords)

all_proposals = []

for _, route_row in inefficient_topk.iterrows():
    route_id = str(route_row['노선번호'])
    print(f'\n{"="*60}')
    print(f'--- 노선 {route_id} ({route_row.get("노선유형","")}) ---')
    
    # 해당 노선 정류소
    route_stops = bus_routes_gdf[bus_routes_gdf['노선번호'] == route_id].sort_values('정류소순번').copy()
    route_stops['정류소ID'] = route_stops['정류소ID'].astype(str)
    
    if len(route_stops) == 0:
        print('  정류소 없음. 스킵.')
        continue
    
    # 정류소별 승차량 (매핑된 ID 사용)
    route_boarding = bus_boarding_only[bus_boarding_only['transport_name'].astype(str) == route_id]
    
    if len(route_boarding) == 0:
        print(f'  승차 데이터 없음. 스킵.')
        continue
    
    stop_usage = route_boarding.groupby(STATION_KEY).size().reset_index(name='총승차량')
    stop_usage.rename(columns={STATION_KEY: 'matched_id'}, inplace=True)
    stop_usage['일평균승차량'] = stop_usage['총승차량'] / total_days
    
    # 정류소와 병합
    route_stops_demand = route_stops.merge(
        stop_usage, left_on='정류소ID', right_on='matched_id', how='left'
    )
    route_stops_demand['일평균승차량'] = route_stops_demand['일평균승차량'].fillna(0)
    
    mean_demand = route_stops_demand['일평균승차량'].mean()
    std_demand = route_stops_demand['일평균승차량'].std()
    threshold = max(mean_demand - 2 * std_demand, 1)  # 최소 1명
    
    print(f'  정류소 수: {len(route_stops)}')
    print(f'  일평균 승차량: mean={mean_demand:.1f}, std={std_demand:.1f}')
    print(f'  제거/이동 기준: mean - 2*std = {threshold:.1f}명/일')
    
    # ----- 제거/이동 판단 -----
    low_demand = route_stops_demand[route_stops_demand['일평균승차량'] < threshold]
    print(f'  기준 미달 정류소: {len(low_demand)}개')
    
    for _, stop in low_demand.iterrows():
        sx, sy = stop.geometry.x, stop.geometry.y
        
        # 주변 300m 이내 사회형평 시설 있는지 확인
        has_equity = False
        for ep_gdf in equity_points:
            if len(ep_gdf) > 0:
                dists = ep_gdf.geometry.distance(stop.geometry)
                if (dists < 300).any():
                    has_equity = True
                    break
        
        # 주변 400m 격자 내 생활인구 확인
        nearby_pop = 0
        if POP_COL and POP_COL in grid_merged.columns:
            gm_dists = np.sqrt((grid_merged['cx'] - sx)**2 + (grid_merged['cy'] - sy)**2)
            nearby_grids = grid_merged[gm_dists <= 400]
            nearby_pop = nearby_grids[POP_COL].sum() if len(nearby_grids) > 0 else 0
        
        if has_equity:
            # 사회형평 시설 근처 -> 유지 (이동하지 않음)
            all_proposals.append({
                '노선번호': route_id, '변경유형': '유지(형평)',
                '정류소ID': stop['정류소ID'], '정류소명': stop['정류소명'],
                'X좌표': sx, 'Y좌표': sy,
                '변경사유': f'일평균 {stop["일평균승차량"]:.1f}명이나 사회형평시설 300m내',
                '일평균승차량': stop['일평균승차량'],
                '신규X': np.nan, '신규Y': np.nan, '신규노드ID': None
            })
            continue
        
        if nearby_pop > 0 and nearby_pop > mean_demand * 0.5:
            # 생활인구는 있지만 승차량 낮음 -> 이동 후보
            # 주변 800m 이내에서 생활인구 밀도가 가장 높은 지점 탐색
            if POP_COL and POP_COL in grid_merged.columns:
                gm_dists = np.sqrt((grid_merged['cx'] - sx)**2 + (grid_merged['cy'] - sy)**2)
                candidate_grids = grid_merged[(gm_dists > 200) & (gm_dists <= 800)].copy()
                
                if len(candidate_grids) > 0 and POP_COL in candidate_grids.columns:
                    # 생활인구 밀도 상위 지점
                    best_grid = candidate_grids.nlargest(1, POP_COL).iloc[0]
                    target_x, target_y = best_grid['cx'], best_grid['cy']
                    
                    # 가장 가까운 차량 노드로 스냅
                    _, node_idx = car_node_tree.query([target_x, target_y])
                    snap_node_id = car_node_ids[node_idx]
                    snap_x = G_car.nodes[snap_node_id].get('x', target_x)
                    snap_y = G_car.nodes[snap_node_id].get('y', target_y)
                    
                    all_proposals.append({
                        '노선번호': route_id, '변경유형': '이동',
                        '정류소ID': stop['정류소ID'], '정류소명': stop['정류소명'],
                        'X좌표': sx, 'Y좌표': sy,
                        '변경사유': f'일평균 {stop["일평균승차량"]:.1f}명 → 생활인구 밀집 지역(노드 {snap_node_id})으로 이동',
                        '일평균승차량': stop['일평균승차량'],
                        '신규X': snap_x, '신규Y': snap_y, '신규노드ID': snap_node_id
                    })
                    continue
        
        # 생활인구도 낮고 형평시설도 없음 -> 제거
        all_proposals.append({
            '노선번호': route_id, '변경유형': '제거',
            '정류소ID': stop['정류소ID'], '정류소명': stop['정류소명'],
            'X좌표': sx, 'Y좌표': sy,
            '변경사유': f'일평균 {stop["일평균승차량"]:.1f}명 (기준 {threshold:.1f}명 미만), 주변 생활인구 부족',
            '일평균승차량': stop['일평균승차량'],
            '신규X': np.nan, '신규Y': np.nan, '신규노드ID': None
        })
    
    # ----- 신규 정류소 추가: 생활인구+사회형평 기반 -----
    route_buffer_400 = unary_union(route_stops.geometry.buffer(400))
    route_buffer_1200 = unary_union(route_stops.geometry.buffer(1200))
    
    # 400~1200m 범위에서 생활인구 밀도 높은 격자 탐색
    if POP_COL and POP_COL in grid_merged.columns:
        grid_centroids = grid_merged[['cx','cy']].values
        in_1200 = np.array([route_buffer_1200.contains(Point(x,y)) for x,y in grid_centroids])
        in_400 = np.array([route_buffer_400.contains(Point(x,y)) for x,y in grid_centroids])
        gap_mask = in_1200 & ~in_400
        
        gap_grids_local = grid_merged[gap_mask].copy()
        if len(gap_grids_local) > 0:
            # 상위 인구밀도 격자
            top_grids = gap_grids_local.nlargest(3, POP_COL)
            for _, tg in top_grids.iterrows():
                # 차량 노드로 스냅
                _, nidx = car_node_tree.query([tg['cx'], tg['cy']])
                new_node = car_node_ids[nidx]
                new_x = G_car.nodes[new_node].get('x', tg['cx'])
                new_y = G_car.nodes[new_node].get('y', tg['cy'])
                
                all_proposals.append({
                    '노선번호': route_id, '변경유형': '신설',
                    '정류소ID': f'NEW_{new_node}', '정류소명': f'신규정류소(노드{new_node})',
                    'X좌표': new_x, 'Y좌표': new_y,
                    '변경사유': f'생활인구 밀집 공백지역, 노드 {new_node}에 배치',
                    '일평균승차량': 0,
                    '신규X': new_x, '신규Y': new_y, '신규노드ID': new_node
                })

proposals_df = pd.DataFrame(all_proposals)
print(f'\n{"="*60}')
print(f'=== 전체 변경 제안 요약 ===')
print(f'총 제안: {len(proposals_df)}건')
if len(proposals_df) > 0:
    print(proposals_df['변경유형'].value_counts())
    print(f'\n노선별 요약:')
    for rid, grp in proposals_df.groupby('노선번호'):
        print(f'  {rid}: 제거 {(grp["변경유형"]=="제거").sum()}, 이동 {(grp["변경유형"]=="이동").sum()}, '
              f'신설 {(grp["변경유형"]=="신설").sum()}, 유지(형평) {(grp["변경유형"]=="유지(형평)").sum()}')


In [ ]:
# ===========================================================
# 시각화: 노선별 정류장 제거/이동/신설 지도 (노드링크 기반)
# ===========================================================
print('비효율 노선 변경 제안 시각화 중...')

for _, route_row in inefficient_topk.head(5).iterrows():
    route_id = str(route_row['노선번호'])
    route_proposals = proposals_df[proposals_df['노선번호'] == route_id]
    
    if len(route_proposals) == 0:
        continue
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 11))
    
    # 노선 정류소
    route_stops_viz = bus_routes_gdf[bus_routes_gdf['노선번호'] == route_id]
    if len(route_stops_viz) == 0:
        plt.close()
        continue
    
    bounds = route_stops_viz.total_bounds
    margin = 2000
    ax.set_xlim(bounds[0]-margin, bounds[2]+margin)
    ax.set_ylim(bounds[1]-margin, bounds[3]+margin)
    
    # 배경
    census.plot(ax=ax, color='#f5f5f5', edgecolor='#dddddd', linewidth=0.2)
    
    # 도로 네트워크 (배경)
    car_link_clip = car_link.cx[bounds[0]-margin:bounds[2]+margin, bounds[1]-margin:bounds[3]+margin]
    if len(car_link_clip) > 0:
        car_link_clip.plot(ax=ax, color='#e0e0e0', linewidth=0.3, alpha=0.5)
    
    # 기존 경로 (회색)
    if route_id in route_lines:
        gpd.GeoDataFrame(geometry=[route_lines[route_id]], crs=TARGET_CRS).plot(
            ax=ax, color='gray', linewidth=2.5, alpha=0.6, label='기존 경로')
    
    # 기존 정류소 (회색 원)
    route_stops_viz.plot(ax=ax, color='gray', markersize=25, zorder=5, alpha=0.5)
    
    # 제거 정류소 (빨간 X)
    removes = route_proposals[route_proposals['변경유형'] == '제거']
    if len(removes) > 0:
        ax.scatter(removes['X좌표'], removes['Y좌표'],
                   color='red', marker='x', s=120, zorder=10, linewidths=2.5, label=f'제거 ({len(removes)}개)')
    
    # 이동 정류소 (주황 화살표)
    moves = route_proposals[route_proposals['변경유형'] == '이동']
    if len(moves) > 0:
        for _, mv in moves.iterrows():
            if pd.notna(mv['신규X']) and pd.notna(mv['신규Y']):
                ax.annotate('', xy=(mv['신규X'], mv['신규Y']), xytext=(mv['X좌표'], mv['Y좌표']),
                            arrowprops=dict(arrowstyle='->', color='orange', lw=2))
                ax.scatter(mv['신규X'], mv['신규Y'], color='orange', marker='D', s=80, zorder=10)
        ax.scatter([], [], color='orange', marker='D', s=80, label=f'이동 ({len(moves)}개)')
    
    # 신설 정류소 (파란 별)
    news = route_proposals[route_proposals['변경유형'] == '신설']
    if len(news) > 0:
        ax.scatter(news['X좌표'], news['Y좌표'],
                   color='blue', marker='*', s=200, zorder=10, label=f'신설 ({len(news)}개)')
    
    # 유지(형평) 정류소 (초록 삼각)
    keeps = route_proposals[route_proposals['변경유형'] == '유지(형평)']
    if len(keeps) > 0:
        ax.scatter(keeps['X좌표'], keeps['Y좌표'],
                   color='green', marker='^', s=100, zorder=10, label=f'유지-형평 ({len(keeps)}개)')
    
    ax.set_title(f'노선 {route_id} 정류장 변경 제안\n(기준: mean-2std={route_proposals["일평균승차량"].mean():.1f}명/일)', fontsize=14)
    ax.legend(loc='upper right', fontsize=10)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.savefig(f'{RESULT_DIR}\\route_{route_id}_proposal.png', dpi=150, bbox_inches='tight')
    plt.show()

print('시각화 완료')


---
## Part 3. 시나리오 2 - 심야 버스 노선 신설

### 3-1. 심야 수요 분석

In [ ]:
# ===========================================================
# 심야(0~5시) 교통카드 이용량 분석
# ===========================================================
print('=== 심야 수요 분석 (0~5시) ===')

night_card = card_df[card_df['심야여부']].copy()
night_bus = night_card[night_card['수단'] == '버스'].copy()
night_subway = night_card[night_card['수단'] == '지하철'].copy()

print(f'심야 전체 거래: {len(night_card):,}건')
print(f'  버스: {len(night_bus):,}건')
print(f'  지하철: {len(night_subway):,}건')

# 기존 심야 버스 노선 존재 여부
night_routes = night_bus[night_bus['transport_name'].astype(str).str.contains('심야', na=False)]
existing_night_routes = night_routes['transport_name'].unique()
print(f'\n기존 심야 버스 노선: {len(existing_night_routes)}개')
for r in existing_night_routes:
    cnt = (night_routes['transport_name'] == r).sum()
    print(f'  {r}: {cnt:,}건')

# 심야 비-심야노선 이용현황
non_night_routes_usage = night_bus[~night_bus['transport_name'].astype(str).str.contains('심야', na=False)]
print(f'\n심야 시간 비-심야노선 이용: {len(non_night_routes_usage):,}건')
print(f'  이용 노선 수: {non_night_routes_usage["transport_name"].nunique()}')

In [ ]:
# 심야 시간대별 이용량 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 시간대별 (0~5시 세분화)
night_hourly = night_card.groupby(['시간대', '수단']).size().unstack(fill_value=0)
night_hourly.plot(kind='bar', ax=axes[0], width=0.7)
axes[0].set_title('심야 시간대별 이용량 (0~5시)', fontsize=13)
axes[0].set_xlabel('시간대')
axes[0].set_ylabel('거래 건수')
axes[0].tick_params(axis='x', rotation=0)

# 심야 버스 상위 정류소
night_bus_boarding = night_bus[night_bus['승하차'] == '승차']
top_night_stops = night_bus_boarding.groupby('station_name').size().nlargest(15)
top_night_stops.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('심야 버스 상위 승차 정류소 Top 15', fontsize=13)
axes[1].set_xlabel('승차 건수')

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}\\fig_night_demand.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 심야 생활인구 + 카드소비 연계 분석
print('=== 심야 생활인구 + 카드소비 연계 ===')

# 심야 카드소비 (시간대 0~5)
night_consume = card_consume_tme[card_consume_tme['tme'].isin([0, 1, 2, 3, 4, 5])]
night_consume_grid = night_consume.groupby('id').agg(
    심야소비금액=('amt', 'sum'),
    심야소비건수=('cnt', 'sum')
).reset_index()

print(f'심야 카드소비 격자 수: {len(night_consume_grid):,}')
print(f'심야 총 소비금액: {night_consume_grid["심야소비금액"].sum():,.0f}원')

# 격자에 심야 데이터 병합 (타입 맞춤)
grid_night_pop['id'] = grid_night_pop['id'].astype(str)  # int64 -> str
night_consume_grid['id'] = night_consume_grid['id'].astype(str)  # int64 -> str

grid_night = grid_50m.merge(grid_night_pop, on='id', how='left')
grid_night = grid_night.merge(night_consume_grid, on='id', how='left')
grid_night['평균심야인구'] = grid_night['평균심야인구'].fillna(0)
grid_night['심야소비금액'] = grid_night['심야소비금액'].fillna(0)

# 심야 수요 종합 점수
grid_night['심야수요점수'] = (
        normalize(grid_night['평균심야인구']) * 0.6 +
        normalize(grid_night['심야소비금액']) * 0.4
)

print(f'심야 수요 점수 통계:')
print(grid_night['심야수요점수'].describe().round(4))


### 3-2. 심야 서비스 공백 분석

In [ ]:
# ===========================================================
# 기존 심야 노선 커버리지 (400m 버퍼)
# ===========================================================
print('=== 심야 서비스 공백 분석 ===')

# 기존 심야 노선 정류소
night_route_stops = bus_routes_gdf[
    bus_routes_gdf['노선번호'].astype(str).str.contains('심야')
]

if len(night_route_stops) > 0:
    night_buffer = unary_union(night_route_stops.geometry.buffer(400))
    print(f'기존 심야 노선 정류소 수: {len(night_route_stops):,}')
    
    # 커버 영역 내/외 격자 분류
    grid_night['심야커버'] = grid_night.geometry.intersects(night_buffer)
else:
    print('기존 심야 노선 정류소 없음 (노선명 매칭 실패 가능)')
    # transport_name 기반으로 재탐색
    night_transport_names = [str(r) for r in existing_night_routes]
    night_route_stops_alt = bus_routes_gdf[
        bus_routes_gdf['노선번호'].isin(night_transport_names)
    ]
    if len(night_route_stops_alt) > 0:
        night_route_stops = night_route_stops_alt
        night_buffer = unary_union(night_route_stops.geometry.buffer(400))
        grid_night['심야커버'] = grid_night.geometry.intersects(night_buffer)
        print(f'대안 매칭: {len(night_route_stops):,}개 정류소')
    else:
        grid_night['심야커버'] = False
        night_buffer = None
        print('심야 노선 정류소 매칭 불가 - 전체를 공백으로 처리')

# 심야 수요 있지만 서비스 안 되는 공백 지역
# 수요 있음 = 심야수요점수 상위 30%
demand_threshold = grid_night['심야수요점수'].quantile(0.70)
grid_night['수요있음'] = grid_night['심야수요점수'] >= demand_threshold
grid_night['공백지역'] = grid_night['수요있음'] & ~grid_night['심야커버']

print(f'\n심야 수요 있는 격자: {grid_night["수요있음"].sum():,}개')
print(f'기존 서비스 커버 격자: {grid_night["심야커버"].sum():,}개')
print(f'공백 지역 (수요 있지만 서비스 없음): {grid_night["공백지역"].sum():,}개')

In [ ]:
# 공백 지역 시각화
fig, ax = plt.subplots(1, 1, figsize=(14, 12))

# 배경
census.plot(ax=ax, color='#f5f5f5', edgecolor='#dddddd', linewidth=0.2)

# 공백 지역 표시
gap_grids = grid_night[grid_night['공백지역']]
if len(gap_grids) > 0:
    gap_grids.plot(ax=ax, color='red', alpha=0.4, label='공백 지역')

# 기존 심야 노선
if len(night_route_stops) > 0:
    night_route_stops.plot(ax=ax, color='blue', markersize=10, alpha=0.6, label='기존 심야 정류소')

ax.set_title('심야 서비스 공백 지역 (수요 상위 30% 중 미커버)', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.set_axis_off()

plt.tight_layout()
plt.savefig(f'{RESULT_DIR}\\fig_night_gap.png', dpi=150, bbox_inches='tight')
plt.show()

### 3-3. 심야 노선 제안

In [ ]:
# ===========================================================
# 공백 지역 수요 클러스터링 (DBSCAN) - 노선별 분리용
# ===========================================================
print('=== 심야 공백 지역 클러스터링 (DBSCAN) ===')

gap_centroids = gap_grids.copy()
gap_centroids['cx'] = gap_centroids.geometry.centroid.x
gap_centroids['cy'] = gap_centroids.geometry.centroid.y

coords = gap_centroids[['cx', 'cy']].values

# DBSCAN: eps=800m (노선 내 클러스터를 좀 더 넓게), min_samples=8
db = DBSCAN(eps=800, min_samples=8)
gap_centroids['cluster'] = db.fit_predict(coords)

clusters = gap_centroids[gap_centroids['cluster'] >= 0]
n_clusters = clusters['cluster'].nunique()

print(f'클러스터 수: {n_clusters}')
print(f'노이즈 포인트: {(gap_centroids["cluster"] == -1).sum()}')

cluster_info = clusters.groupby('cluster').agg(
    격자수=('id', 'count'),
    평균수요=('심야수요점수', 'mean'),
    총수요=('심야수요점수', 'sum'),
    중심X=('cx', 'mean'),
    중심Y=('cy', 'mean')
).reset_index()
cluster_info = cluster_info.sort_values('총수요', ascending=False)

print(f'\n클러스터별 정보 (상위 20):')
print(cluster_info.head(20).to_string(index=False))

In [ ]:
# ===========================================================
# 심야 노선 설계 - 5개 (노선별 개별 설계, 노드-링크 매핑)
# ===========================================================
print('=== 심야 노선 설계 (5개, 노드-링크 기반 실제 도로 매핑) ===')

# 정류소 KDTree
stop_tree_night = cKDTree(
    np.column_stack([bus_stops_shp.geometry.x.values, bus_stops_shp.geometry.y.values])
)

# 도로 노드 KDTree
if 'car_node_tree' not in dir():
    car_node_coords = np.array([(G_car.nodes[n].get('x',0), G_car.nodes[n].get('y',0)) for n in G_car.nodes()])
    car_node_ids = list(G_car.nodes())
    car_node_tree = cKDTree(car_node_coords)

# --- 클러스터를 5개 노선 그룹으로 ---
from sklearn.cluster import KMeans

N_ROUTES = 5

if n_clusters <= N_ROUTES:
    cluster_info['route_group'] = range(len(cluster_info))
    N_ROUTES = len(cluster_info)
else:
    km = KMeans(n_clusters=N_ROUTES, random_state=42, n_init=10)
    cluster_info['route_group'] = km.fit_predict(
        cluster_info[['중심X', '중심Y']].values
    )

print(f'심야 노선 수: {N_ROUTES}개')
for rg in range(N_ROUTES):
    members = cluster_info[cluster_info['route_group'] == rg]
    print(f'  노선 {rg+1}: {len(members)}개 클러스터, 총수요 {members["총수요"].sum():.1f}')


def design_night_route_v2(route_clusters, route_name, G, car_node_tree, car_node_ids, 
                           stop_tree, stops_shp, car_link_gdf):
    """
    심야 노선 1개 설계 (v2 - 노드-링크 매핑 기반)
    
    1) 클러스터 중심 -> 가장 가까운 기존 정류소
    2) TSP(Nearest Neighbor) 순서 결정
    3) 연속 정류소 간 도로 네트워크 최단경로로 연결
    4) 경로 상 1km 간격으로 기존 정류소 스냅
    5) 경로 geometry = 실제 도로 링크 연결
    """
    # 1) 클러스터별 대표 정류소
    key_stops = []
    for _, cl in route_clusters.iterrows():
        _, idx = stop_tree.query([cl['중심X'], cl['중심Y']])
        stop = stops_shp.iloc[idx]
        key_stops.append({
            'cluster': cl['cluster'],
            'bstopid': str(stop['bstopid']),
            'bstopnm': stop['bstopnm'],
            'x': stop.geometry.x, 'y': stop.geometry.y,
            '수요': cl['총수요']
        })
    key_df = pd.DataFrame(key_stops).drop_duplicates(subset='bstopid')
    
    # 정류소가 2개 미만이면 주변 보강
    if len(key_df) < 3:
        cx, cy = key_df['x'].mean(), key_df['y'].mean()
        _, nearby_idxs = stop_tree.query([cx, cy], k=8)
        existing_ids = set(key_df['bstopid'])
        for ni in nearby_idxs:
            s = stops_shp.iloc[ni]
            if str(s['bstopid']) not in existing_ids:
                key_df = pd.concat([key_df, pd.DataFrame([{
                    'cluster': -1, 'bstopid': str(s['bstopid']), 'bstopnm': s['bstopnm'],
                    'x': s.geometry.x, 'y': s.geometry.y, '수요': 0
                }])], ignore_index=True)
                existing_ids.add(str(s['bstopid']))
            if len(key_df) >= 5:
                break
    
    # 2) Nearest Neighbor TSP
    points = key_df[['x','y']].values
    n = len(points)
    visited = [False] * n
    # 수요 가장 높은 곳에서 시작
    start_idx = key_df['수요'].idxmax() if '수요' in key_df.columns else 0
    start_pos = key_df.index.get_loc(start_idx) if start_idx in key_df.index else 0
    order = [start_pos]
    visited[start_pos] = True
    
    for _ in range(n - 1):
        curr = order[-1]
        best_d, best_j = float('inf'), -1
        for j in range(n):
            if not visited[j]:
                d = np.sqrt((points[curr][0]-points[j][0])**2 + (points[curr][1]-points[j][1])**2)
                if d < best_d:
                    best_d, best_j = d, j
        if best_j >= 0:
            order.append(best_j)
            visited[best_j] = True
    
    ordered = key_df.iloc[order].reset_index(drop=True)
    
    # 3) 노드-링크 최단경로로 연결 + 경로 geometry 수집
    route_link_ids = []
    route_geometry_parts = []
    total_length = 0
    all_waypoint_stops = []
    
    for i in range(len(ordered) - 1):
        src = ordered.iloc[i]
        dst = ordered.iloc[i + 1]
        
        # 각 정류소에서 가장 가까운 도로 노드 찾기
        _, src_node_idx = car_node_tree.query([src['x'], src['y']])
        _, dst_node_idx = car_node_tree.query([dst['x'], dst['y']])
        src_node = car_node_ids[src_node_idx]
        dst_node = car_node_ids[dst_node_idx]
        
        try:
            path_nodes = nx.shortest_path(G, src_node, dst_node, weight='weight')
            path_length = nx.shortest_path_length(G, src_node, dst_node, weight='weight')
            total_length += path_length
        except nx.NetworkXNoPath:
            # 경로 없으면 직선 연결
            direct_dist = np.sqrt((src['x']-dst['x'])**2 + (src['y']-dst['y'])**2)
            total_length += direct_dist
            route_geometry_parts.append(LineString([(src['x'],src['y']), (dst['x'],dst['y'])]))
            continue
        
        # 경로 노드 좌표로 LineString 생성
        path_coords = []
        for nd in path_nodes:
            nx_data = G.nodes[nd]
            path_coords.append((nx_data.get('x', 0), nx_data.get('y', 0)))
        
        if len(path_coords) >= 2:
            route_geometry_parts.append(LineString(path_coords))
        
        # 경로 상 링크 ID 수집
        for j in range(len(path_nodes) - 1):
            edge_data = G.get_edge_data(path_nodes[j], path_nodes[j+1], default={})
            if 'LINK_ID' in edge_data:
                route_link_ids.append(edge_data['LINK_ID'])
        
        # 4) 경로 상 1km 간격으로 기존 정류소 스냅
        cum_dist = 0
        last_snap = 0
        for j in range(1, len(path_coords)):
            seg_dist = np.sqrt((path_coords[j][0]-path_coords[j-1][0])**2 + 
                               (path_coords[j][1]-path_coords[j-1][1])**2)
            cum_dist += seg_dist
            
            if cum_dist - last_snap >= 1000:  # 1km 간격
                _, snap_idx = stop_tree.query(path_coords[j])
                snap_stop = stops_shp.iloc[snap_idx]
                snap_dist = np.sqrt((snap_stop.geometry.x - path_coords[j][0])**2 +
                                    (snap_stop.geometry.y - path_coords[j][1])**2)
                
                if snap_dist <= 500:  # 500m 이내 기존 정류소만
                    all_waypoint_stops.append({
                        'bstopid': str(snap_stop['bstopid']),
                        'bstopnm': snap_stop['bstopnm'],
                        'x': snap_stop.geometry.x,
                        'y': snap_stop.geometry.y,
                        '유형': '경유'
                    })
                    last_snap = cum_dist
    
    # 전체 경로 geometry 합치기
    if route_geometry_parts:
        route_line = MultiLineString(route_geometry_parts) if len(route_geometry_parts) > 1 else route_geometry_parts[0]
    else:
        coords = [(row['x'], row['y']) for _, row in ordered.iterrows()]
        route_line = LineString(coords) if len(coords) >= 2 else None
    
    # 전체 정류소 (주요 + 경유)
    wp_df = pd.DataFrame(all_waypoint_stops).drop_duplicates(subset='bstopid') if all_waypoint_stops else pd.DataFrame()
    
    key_for_merge = ordered[['bstopid','bstopnm','x','y']].copy()
    key_for_merge['유형'] = '주요'
    
    if len(wp_df) > 0:
        # 주요 정류소와 겹치는 경유 정류소 제거
        wp_df = wp_df[~wp_df['bstopid'].isin(set(key_for_merge['bstopid']))]
        all_stops = pd.concat([key_for_merge, wp_df], ignore_index=True)
    else:
        all_stops = key_for_merge.copy()
    
    return {
        'name': route_name,
        'route_line': route_line,
        'route_link_ids': route_link_ids,
        'length_km': total_length / 1000,
        'key_stops': ordered,
        'all_stops': all_stops,
        'n_key': len(ordered),
        'n_total': len(all_stops)
    }


# --- 5개 노선 설계 (노선별 개별) ---
proposed_routes = []

for rg in range(N_ROUTES):
    route_name = f'심야신설{rg+1}호'
    members = cluster_info[cluster_info['route_group'] == rg].copy()
    
    if len(members) == 0:
        continue
    
    print(f'\n--- {route_name} 설계 중 ({len(members)}개 클러스터) ---')
    
    route_result = design_night_route_v2(
        members, route_name, G_car, car_node_tree, car_node_ids,
        stop_tree_night, bus_stops_shp, car_link
    )
    proposed_routes.append(route_result)
    
    print(f'  경로 길이: {route_result["length_km"]:.2f} km')
    print(f'  주요 정류소: {route_result["n_key"]}개, 전체: {route_result["n_total"]}개')
    print(f'  사용 도로 링크: {len(route_result["route_link_ids"])}개')
    print(f'  주요 경유지:')
    for _, s in route_result['key_stops'].iterrows():
        print(f'    - {s["bstopnm"]} (수요: {s.get("수요", "-")})')
    
    # 경유 정류소 목록
    waypoints = route_result['all_stops'][route_result['all_stops']['유형'] == '경유']
    if len(waypoints) > 0:
        print(f'  경유 정류소 ({len(waypoints)}개):')
        for _, w in waypoints.iterrows():
            print(f'    - {w["bstopnm"]}')

print(f'\n{"="*60}')
print(f'=== 심야 노선 설계 요약 ===')
for pr in proposed_routes:
    print(f'{pr["name"]}: {pr["length_km"]:.1f}km, {pr["n_total"]}개 정류소 ({pr["n_key"]}주요 + {pr["n_total"]-pr["n_key"]}경유)')


In [ ]:
# ===========================================================
# 심야 노선 상세 정보 + 기존 노선 중복도 검증
# ===========================================================
print('=== 심야 노선 상세 & 중복도 검증 ===')

# 기존 심야 노선 정류소 ID
night_existing_sids = set()
if len(night_route_stops) > 0:
    night_existing_sids = set(night_route_stops['정류소ID'].astype(str))

# 노선별 상세 정보
night_route_details = []

for pr in proposed_routes:
    route_sids = set(pr['all_stops']['bstopid'].astype(str))
    overlap = route_sids & night_existing_sids
    overlap_rate = len(overlap) / len(route_sids) if route_sids else 0
    
    # 노선 간 중복 체크
    other_sids = set()
    for pr2 in proposed_routes:
        if pr2['name'] != pr['name']:
            other_sids.update(pr2['all_stops']['bstopid'].astype(str))
    inter_overlap = route_sids & other_sids
    inter_rate = len(inter_overlap) / len(route_sids) if route_sids else 0
    
    print(f'\n--- {pr["name"]} ---')
    print(f'  경로 길이: {pr["length_km"]:.2f} km')
    print(f'  정류소 수: {pr["n_total"]}개 (주요 {pr["n_key"]}개)')
    print(f'  기존 심야 중복: {len(overlap)}개 ({overlap_rate:.1%})')
    print(f'  신규 노선간 중복: {len(inter_overlap)}개 ({inter_rate:.1%})')
    
    if overlap_rate >= 0.3:
        print(f'  => 기존 심야 노선과 중복도 높음 - 경로 조정 검토')
    else:
        print(f'  => 신규 노선으로 적합')
    
    print(f'  경유 정류소 목록:')
    for _, s in pr['all_stops'].iterrows():
        marker = ' (주요)' if s['bstopid'] in pr['key_stops']['bstopid'].values else ''
        print(f'    {s["bstopnm"]}{marker}')
    
    night_route_details.append({
        '노선명': pr['name'],
        '경로길이_km': pr['length_km'],
        '정류소수': pr['n_total'],
        '주요정류소수': pr['n_key'],
        '기존중복률': overlap_rate,
        '노선간중복률': inter_rate
    })

# 전체 커버리지 확인
all_proposed_sids = set()
for pr in proposed_routes:
    all_proposed_sids.update(pr['all_stops']['bstopid'].astype(str))
combined_with_existing = all_proposed_sids | night_existing_sids

print(f'\n{"="*50}')
print(f'=== 심야 노선 종합 ===')
print(f'{"="*50}')
print(f'신규 노선 수: {len(proposed_routes)}')
print(f'신규 정류소 수 (중복 제거): {len(all_proposed_sids)}')
print(f'기존 심야 정류소 수: {len(night_existing_sids)}')
print(f'합산 정류소 수: {len(combined_with_existing)}')

# 전체 정류소 DataFrame (결과 저장용)
full_stops_all = pd.concat([pr['all_stops'] for pr in proposed_routes]).drop_duplicates(subset='bstopid')
full_stops_all['노선명'] = None
for pr in proposed_routes:
    mask = full_stops_all['bstopid'].isin(pr['all_stops']['bstopid'])
    # 다중 노선 소속 가능
    existing = full_stops_all.loc[mask, '노선명'].fillna('')
    full_stops_all.loc[mask, '노선명'] = existing.where(existing == '', existing + ',') + pr['name']

# proposed_route_nodes, proposed_route_length, full_stops 호환용 (시각화 셀에서 사용)
# 모든 노선의 노드를 합침
proposed_route_nodes = []
proposed_route_length = sum(pr['length_km'] * 1000 for pr in proposed_routes)
full_stops = full_stops_all

print(f'\n노선별 요약:')
details_df = pd.DataFrame(night_route_details)
print(details_df.to_string(index=False))

In [ ]:
# 기존 노선 중복도 검증
print('=== 기존 노선 중복도 검증 ===')

# 제안 노선 경유 정류소가 기존 심야 노선에 포함되는 비율
proposed_stop_ids = set(full_stops['bstopid'].astype(str))

night_existing_stop_ids = set(
    night_route_stops['정류소ID'].astype(str)
) if len(night_route_stops) > 0 else set()

overlap_count = len(proposed_stop_ids & night_existing_stop_ids)
overlap_rate = overlap_count / len(proposed_stop_ids) if len(proposed_stop_ids) > 0 else 0

print(f'제안 정류소 수: {len(proposed_stop_ids)}')
print(f'기존 심야 노선 정류소 수: {len(night_existing_stop_ids)}')
print(f'중복 정류소: {overlap_count}개 ({overlap_rate*100:.1f}%)')

if overlap_rate < 0.3:
    print('=> 기존 심야 노선과 중복도 낮음 - 신규 노선으로 적합')
else:
    print('=> 기존 심야 노선과 중복도 높음 - 경로 조정 필요')

In [ ]:
# ===========================================================
# 심야 노선 시각화: 노선별 개별 맵 (기존 파랑 + 신규 빨강 + 도로 경로)
# ===========================================================
print('심야 노선 개별 시각화 중...')

for pr in proposed_routes:
    fig, ax = plt.subplots(1, 1, figsize=(14, 12))
    
    # 경로 범위 결정
    all_x = pr['all_stops']['x'].values
    all_y = pr['all_stops']['y'].values
    margin = 3000
    ax.set_xlim(all_x.min()-margin, all_x.max()+margin)
    ax.set_ylim(all_y.min()-margin, all_y.max()+margin)
    
    # 배경
    census.plot(ax=ax, color='#f5f5f5', edgecolor='#dddddd', linewidth=0.2)
    
    # 도로 네트워크 (연한 회색)
    clip_bounds = (all_x.min()-margin, all_y.min()-margin, all_x.max()+margin, all_y.max()+margin)
    car_link_clip = car_link.cx[clip_bounds[0]:clip_bounds[2], clip_bounds[1]:clip_bounds[3]]
    if len(car_link_clip) > 0:
        car_link_clip.plot(ax=ax, color='#e8e8e8', linewidth=0.3, alpha=0.5)
    
    # 기존 심야 노선 (파란색, 가느다란)
    if 'night_route_stops' in dir() and len(night_route_stops) > 0:
        for nr_id, nr_grp in night_route_stops.groupby('노선번호'):
            nr_sorted = nr_grp.sort_values('정류소순번')
            if len(nr_sorted) >= 2:
                coords = list(zip(nr_sorted.geometry.x, nr_sorted.geometry.y))
                ax.plot(*zip(*coords), color='#4a90d9', linewidth=1.5, alpha=0.4)
    
    # 신규 노선 경로 (빨간색, 굵은 실선 - 실제 도로 매핑)
    if pr['route_line'] is not None:
        route_gdf = gpd.GeoDataFrame(geometry=[pr['route_line']], crs=TARGET_CRS)
        route_gdf.plot(ax=ax, color='#e63946', linewidth=3, alpha=0.8, label=f'{pr["name"]} 경로')
    
    # 주요 정류소 (빨간 원, 큰)
    key_stops = pr['all_stops'][pr['all_stops']['유형'] == '주요']
    ax.scatter(key_stops['x'], key_stops['y'], 
               color='#e63946', s=120, zorder=10, edgecolors='white', linewidths=1.5,
               label=f'주요 정류소 ({len(key_stops)}개)')
    
    # 주요 정류소 이름 라벨
    for _, s in key_stops.iterrows():
        ax.annotate(s['bstopnm'], (s['x'], s['y']), 
                    textcoords="offset points", xytext=(8, 8),
                    fontsize=8, color='#e63946', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='#e63946'))
    
    # 경유 정류소 (주황 삼각, 작은)
    way_stops = pr['all_stops'][pr['all_stops']['유형'] == '경유']
    if len(way_stops) > 0:
        ax.scatter(way_stops['x'], way_stops['y'],
                   color='#f4a261', marker='^', s=60, zorder=9, edgecolors='white', linewidths=1,
                   label=f'경유 정류소 ({len(way_stops)}개)')
    
    ax.set_title(f'{pr["name"]} ({pr["length_km"]:.1f}km, {pr["n_total"]}개 정류소)', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig(f'{RESULT_DIR}\\night_route_{pr["name"]}.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'{pr["name"]} 시각화 완료')

print(f'\n전체 심야 노선 시각화 완료')


---
## Part 4. 종합 결과 저장

In [ ]:
# ===========================================================
# 결과 CSV 저장
# ===========================================================
print('=== 결과 저장 ===')

# 1) 노선별 효율성 점수
save_cols = ['노선번호', '노선유형', '정류소수', '노선길이_km', '일평균승차량',
             '수요효율', '커버리지효율', '중복도', 'CV', '첨두비율',
             '사회형평점수', '경제활동밀도', '종합효율점수', '효율순위']
efficiency[save_cols].to_csv(
    f'{RESULT_DIR}\\노선별_효율성점수.csv', index=False, encoding='utf-8-sig'
)
print(f'1) 노선별_효율성점수.csv 저장 완료 ({len(efficiency)}건)')

# 2) 비효율 노선 Top-K
inefficient_topk[save_cols].to_csv(
    f'{RESULT_DIR}\\비효율노선_Top{K}.csv', index=False, encoding='utf-8-sig'
)
print(f'2) 비효율노선_Top{K}.csv 저장 완료 ({len(inefficient_topk)}건)')

# 3) 정류장 변경 제안
proposals_df.to_csv(
    f'{RESULT_DIR}\\정류장_변경제안.csv', index=False, encoding='utf-8-sig'
)
print(f'3) 정류장_변경제안.csv 저장 완료 ({len(proposals_df)}건)')

# 4) 하차 추정 결과 통계
estimation_stats = trip_pairs_estimated['추정방법'].value_counts().reset_index()
estimation_stats.columns = ['추정방법', '건수']
estimation_stats.to_csv(
    f'{RESULT_DIR}\\하차추정_통계.csv', index=False, encoding='utf-8-sig'
)
print(f'4) 하차추정_통계.csv 저장 완료')

# 5) 심야 노선 제안 경유 정류소
full_stops.to_csv(
    f'{RESULT_DIR}\\심야노선_제안_정류소.csv', index=False, encoding='utf-8-sig'
)
print(f'5) 심야노선_제안_정류소.csv 저장 완료 ({len(full_stops)}건)')

# 6) 심야 공백 클러스터 정보
cluster_info.to_csv(
    f'{RESULT_DIR}\\심야_공백_클러스터.csv', index=False, encoding='utf-8-sig'
)
print(f'6) 심야_공백_클러스터.csv 저장 완료 ({len(cluster_info)}건)')

print(f'\n모든 결과가 {RESULT_DIR} 에 저장되었습니다.')

# 저장된 파일 목록
print(f'\n=== 저장 파일 목록 ===')
for f in os.listdir(RESULT_DIR):
    fpath = os.path.join(RESULT_DIR, f)
    size = os.path.getsize(fpath)
    print(f'  {f} ({size/1024:.1f} KB)')